# Analysis of American Parlimentary Debate Association Rounds Fall 2019 - Spring 2026

## Data Curation

The data for this project was sourced at the APDA online forum where, after the conclusion of each tournament, results are posted in the form of PDF Tab Cards. Tab Cards are structured by team where each team has its own table within the PDF. Each table is labelled with the team name, and contains rows that detail the round number, whether the team was in Government or Opposition position, win/loss status, the opponent team name, judge name, the speaks and ranks for each team member, and the total speaks and ranks of the team. For the preliminary data curation, I used pdfplumber to split each PDF into tables in order to extract data for each team. Bounding boxes were used to extract the team names that prefaced each table. Additionally, a reference was necessary to match the team names, which differ by tournament, to the individual debaters in order to create the columns for the opponent names. It was also necessary to collapse the mirrored rows as each match is represented twice, one for the Government teams table and the other in the Opposition team table. Regex patterns were used for cleaning and normalizing the data.  

In [932]:
# imports used for parsing data from PDFs
from pathlib import Path
import re
import logging
import pandas as pd
import pdfplumber

logging.getLogger("pdfminer").setLevel(logging.ERROR)

In [933]:
def get_tab_cards(folder, season, year):
    """
    Creates file paths based on the contents of the data folders. 
    Uses a dictionary to replace file shortenings with the official APDA school name.
    Returns a list for each season's input of the file path, school name, season, and year.
    """
    tab_cards = [] 
    
    for pdf in Path(folder).glob(f"*_{season}{year}_Tab_Card.pdf"):
        school = pdf.name.replace(f"_{season}{year}_Tab_Card.pdf", "")     
        school = {
            "Binghamton": "Binghamton University",
            "BrownAndWesleyan":"Brown + Wesleyan",
            "Chicago":"University of Chicago",
            "ChicagoNortheastern":"University of Chicago + Northeastern",
            "CMU":"Carnegie Mellon",
            "ColumbiaSwarthmore":"Columbia + Swarthmore",
            "CUNYUMD":"CUNY + Maryland",
            "Delaware":"University of Delaware",
            "FranklinAndMarshall": "Franklin and Marshall",
            "Hopkins":"Johns Hopkins",
            "GU":"Georgetown",
            "GW":"George Washington",
            "JHUAU":"Johns Hopkins + American",
            "NU":"Northeastern",
            "NUBC":"Northeastern + Boston College",
            "NYUWashU":"NYU + Washington University",
            "Pitt":"University of Pittsburgh",
            "PittCMU": "University of Pittsburgh + Carnegie Mellon",
            "PrincetonNUBates":"Princeton + Northeastern + Bates",
            "SmithColumbia":"Smith + Columbia",
            "TheCollegeOfNewJersey": "The College of New Jersey",
            "TCNJ": "The College of New Jersey",
            "TempleWesleyan":"Temple + Wesleyan",
            "Tufts2": "Tufts",
            "UMD":"Maryland",
            "UMDBU":"Maryland + Boston University",
            "UMass":"University of Massachusetts",
            "UMass_Amherst":"University of Massachusetts + Amherst",
            "UVA":"University of Virginia",
            "UVAWDS":"University of Virginia + Wellesley",
            "WashU": "Washington University",
            "WestPoint": "West Point",
            "WesleyanTCNJ":"Wesleyan + The College of New Jersey",
            "WilliamAndMary": "William and Mary",
            "William&Mary": "William and Mary",
        }.get(school, school)
        
        tab_cards.append(
            (str(pdf), school, season, year)
        )
    
    return tab_cards

In [934]:
# Initializes tab cards from Fall 2019 - Spring 2026
FALL_2019_TAB_CARDS = get_tab_cards("Fall2019", "Fall", 2019)
SPRING_2020_TAB_CARDS = get_tab_cards("Spring2020", "Spring", 2020)
FALL_2020_TAB_CARDS = get_tab_cards("Fall2020", "Fall", 2020)
SPRING_2021_TAB_CARDS = get_tab_cards("Spring2021", "Spring", 2021)
FALL_2021_TAB_CARDS = get_tab_cards("Fall2021", "Fall", 2021)
SPRING_2022_TAB_CARDS = get_tab_cards("Spring2022", "Spring", 2022)
FALL_2022_TAB_CARDS = get_tab_cards("Fall2022", "Fall", 2022)
SPRING_2023_TAB_CARDS = get_tab_cards("Spring2023", "Spring", 2023)
FALL_2023_TAB_CARDS = get_tab_cards("Fall2023", "Fall", 2023)
SPRING_2024_TAB_CARDS = get_tab_cards("Spring2024", "Spring", 2024)
FALL_2024_TAB_CARDS = get_tab_cards("Fall2024", "Fall", 2024)
SPRING_2025_TAB_CARDS = get_tab_cards("Spring2025", "Spring", 2025)
FALL_2025_TAB_CARDS = get_tab_cards("Fall2025", "Fall", 2025)
SPRING_2026_TAB_CARDS = get_tab_cards("Spring2026", "Spring", 2026)

Several tournaments were excluded for bad formating, lack of tab cards, or lack of permission to the documnets. The following describes those ommited:

In [935]:
# Formatting of the excluded tournaments
from tabulate import tabulate
omitted_tables = [["Fall2019","Fordham, UVA, Columbia","",""],["Spring2020","","GU, Williams",""],
                  ["Fall2020","","Harvard, Swarthmore",""],["Spring2021","Swarthmore","Brandeis, W&M, GU, Yale",""],
                  ["Fall2021","","","Tufts/UMD, GW/Fordham, Brown, Harvard/Penn"],["Spring2022","Princeton, Yale, Rutgers","Brown, Williams","Darthmouth"],
                  ["Fall2022","","Brown, Yale, Williams","Tufts, Harvard, Rutgers"],["Spring2023","Hopkins","Brandeis, Rutgers, Penn, Williams, UChicago","UMass"],
                  ["Fall2023","Hopkins, Swarthmore","Brandeis, Harvard",""],["Spring2024","Amherst","UMass, Brandeis, NYU, Windsor, Tufts",""],
                  ["Fall2024","","Harvard, Binghamton, Tufts, Brown",""],["Spring2025","Rutgers, Amherst, Penn, UVA","","Dartmouth, UT Austin"],
                  ["Fall2025","Drexel","","Brandeis, Bates, American"],["Spring2026","Temple","",""]]
print(tabulate(omitted_tables, headers=["Season","Bad Format", "Lacking Permission", "Missing Tab Card"], tablefmt="grid"))

+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Season     | Bad Format                  | Lacking Permission                          | Missing Tab Card                           |
+============+=============================+=============================================+============================================+
| Fall2019   | Fordham, UVA, Columbia      |                                             |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Spring2020 |                             | GU, Williams                                |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Fall2020   |                             | Har

Below are the regex patterns used for normalization and sanitization as well as the functions used to parse and clean the df:

In [936]:
# pre-compile regex patterns
MULTISPACE_RX = re.compile(r"\s+")
STATUS_PAREN_RX = re.compile(r"\s*\(([NV])\)\s*$")
CLEAN_TRAILING_RX = re.compile(r"\s+$")
TEAM_PREFIX_RX = re.compile(r"Team:\s*(.+)")
EMOJI_RX = re.compile(
    r"[\U00010000-\U0010FFFF"  
    r"\u2300-\u23FF"           
    r"\u2600-\u26FF"           
    r"\u2700-\u27BF"          
    r"\u2B00-\u2BFF"           
    r"\uFE0F"                  
    r"\u200D]"                 
)

NUMBER_RX = re.compile(r"\d")
KEYCAP_RX = re.compile(r"[\u20E3]")
TM_RX = re.compile(r"™")
QUOTES_RX = re.compile(r'"[^"]*"')

Below are the functions used in parsing and cleaning:

In [937]:
def normalize_cell(name):
    """
    Fills NaN with an empty string, removes emojis, and removes spaces
    """
    if name is None:
        return ""

    name = str(name).title()
    name = EMOJI_RX.sub("", name)
    name = KEYCAP_RX.sub("", name)
    name = TM_RX.sub("", name)

    return MULTISPACE_RX.sub(" ", name).strip()

def normalize_names(name):
    """
    Removes numbers
    """
    if name is None:
        return ""

    name = str(name)
    name = NUMBER_RX.sub("", name)
    name = QUOTES_RX.sub("", name)

    return normalize_cell(name)
    
def split_name_and_status(raw_name):
    """
    Return (clean_name, status) extracted from a speaker header.
    """
    if not raw_name:
        return "", None

    name = normalize_names(raw_name)
    match = STATUS_PAREN_RX.search(name)

    if match:
        status = "Novice" if match.group(1) == "N" else "Varsity"
        return STATUS_PAREN_RX.sub("", name).strip(), status

    return name, None

def split_speaks_and_ranks(df, score_col, prefix):
    """
    Split a score column into separate speaks and rank columns.
    """
    if score_col not in df.columns:
        return df

    cleaned_series = df[score_col].astype(str).str.replace(r"[\(\)\s]", "", regex=True)
    split_data = cleaned_series.str.split(",", expand=True)

    if split_data.shape[1] < 2:
        split_data = pd.DataFrame(index=df.index, columns=[0, 1])

    df[f"{prefix} Speaks"] = pd.to_numeric(split_data[0], errors="coerce")
    df[f"{prefix} Rank"] = pd.to_numeric(split_data[1], errors="coerce").astype("Int64") # Capital I allows NaN integers

    return df.drop(columns=[score_col])

def add_speaker_and_opponent_names(all_df, speaker_df):
    """
    Matches teams and opponents using normalized team keys, then splits
    speaker scores into speaks and rank columns.
    """
    speaker_df = speaker_df.copy()
    speaker_df["_key"] = speaker_df["Team"].map(normalize_names)
    speaker_df = speaker_df.drop_duplicates(subset="_key", keep="first")

    lookup = speaker_df.set_index("_key")[
        ["Speaker One Name", "Speaker One Status", "Speaker Two Name", "Speaker Two Status"]
    ]

    opponent_lookup = lookup.rename(columns={
        "Speaker One Name": "Opponent One Name",
        "Speaker One Status": "Opponent One Status",
        "Speaker Two Name": "Opponent Two Name",
        "Speaker Two Status": "Opponent Two Status"
    })
    
    df = all_df.copy()
    df["Judge"] = df["Judge"].map(normalize_names)
    df["_team_key"] = df["Team"].map(normalize_names)
    df["_opponent_key"] = df["Opponent"].map(normalize_names)
  
    df = df.merge(lookup, left_on="_team_key", right_index=True, how="left")
    df = df.merge(opponent_lookup, left_on="_opponent_key", right_index=True, how="left")


    score_lookup = all_df.copy()
    score_lookup["_opponent_key"] = score_lookup["Team"].map(normalize_names)
    
    score_lookup = (
        score_lookup[
            ["_opponent_key", "Round", "Speaker One Score", "Speaker Two Score"]
        ]
        .drop_duplicates(["_opponent_key", "Round"])
        .rename(columns={
            "Speaker One Score": "Opponent Speaker One Score",
            "Speaker Two Score": "Opponent Speaker Two Score"
        })
    )
    df = df.merge(score_lookup, on=["_opponent_key", "Round"], how="left")
    df = df.drop(columns=["Team", "Opponent", "_team_key", "_opponent_key"])

    df = split_speaks_and_ranks(df, "Speaker One Score", "Speaker One")
    df = split_speaks_and_ranks(df, "Speaker Two Score", "Speaker Two")
    df = split_speaks_and_ranks(df, "Opponent Speaker One Score", "Opponent One")
    df = split_speaks_and_ranks(df, "Opponent Speaker Two Score", "Opponent Two")
    
    ordered_cols = [c for c in [
        "Tournament", "Season", "Year", "Round", "G/O", "W/L",
        "Speaker One Name", "Speaker One Status", "Speaker Two Name", "Speaker Two Status",
        "Opponent One Name", "Opponent One Status", "Opponent Two Name", "Opponent Two Status",
        "Judge", "Speaker One Speaks", "Speaker One Rank", "Speaker Two Speaks", "Speaker Two Rank",
        "Opponent One Speaks", "Opponent One Rank", "Opponent Two Speaks", "Opponent Two Rank", "Total",
    ] if c in df.columns]
    
    return df[ordered_cols]

def parse_tab_cards(card):
    """
    Parses the individual tournament tab card
    """
    all_rows, speaker_lookup = [], []
    seen_teams = set()
    current_team, current_speakers, pending_label = None, None, None
    
    with pdfplumber.open(card) as pdf:
        for page in pdf.pages:
            team_labels = [] # dictionaries for team names + locations
            for line in page.extract_text_lines():
                match = TEAM_PREFIX_RX.search(line["text"])
                if match:
                    team_labels.append({"name": match.group(1).strip(), "top": line["top"]})
 
            tables = page.find_tables()
 
            for table in tables:
                data = table.extract()
                if not data or len(data) < 1:
                    continue
    
                first_cell = normalize_cell(data[0][0]) if data[0] else ""
                has_header = (first_cell in ("R", "")) # stores whether the first cell is the beginning of the table 
 
                if has_header: # if a new table, store 
                    header = [normalize_cell(c) for c in data[0]]
                    body = data[1:]
                    speaker1_header = header[5] if len(header) > 5 else ""
                    speaker2_header = header[6] if len(header) > 6 else ""
                else: # if a continuation of the last table
                    body = data
                    speaker1_header, speaker2_header = current_speakers if current_speakers else ("", "")

                # finds the team name stored in team_labels that is above current table
                table_top = table.bbox[1]
                labels_above = [t for t in team_labels if t["top"] <= table_top]
 
                if labels_above:
                    team_name = normalize_cell(
                        max(labels_above, key=lambda t: t["top"])["name"]
                    )
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                elif has_header and current_speakers and (speaker1_header, speaker2_header) == current_speakers: # handles table breaking to next page with label headings
                    team_name = current_team
                elif not has_header and current_team is not None: # handles continuation table w/o header
                    team_name = current_team
                elif pending_label is not None: # detected team name before
                    team_name = pending_label
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                else:
                    team_name = current_team
                    current_speakers = (speaker1_header, speaker2_header)

                if team_name not in seen_teams:
                    seen_teams.add(team_name) # set for O(1) lookup
                    name1, status1 = split_name_and_status(speaker1_header)
                    name2, status2 = split_name_and_status(speaker2_header)
                    speaker_lookup.append({
                        "Team": team_name,
                        "Speaker One Name": name1, "Speaker One Status": status1,
                        "Speaker Two Name": name2, "Speaker Two Status": status2,
                    })
 
                for row in body:
                    row = [normalize_cell(c) for c in row]
                    if not row or row[0].lower().startswith("tournament totals"):
                        continue  
 
                    row = (row + [""] * 8)[:8]
                    round_no, g_o, w_l, opponent, judge, sp1, sp2, total = row
 
                    if not round_no:
                        if not any([g_o, w_l, opponent, judge, sp1, sp2]):
                            continue
                        round_no = "UNKNOWN (split across page break)"
 
                    all_rows.append({
                        "Team": team_name, "Round": round_no, "G/O": g_o, "W/L": w_l,
                        "Opponent": opponent, "Judge": judge, 
                        "Speaker One Score": sp1, "Speaker Two Score": sp2, "Total": total,
                    })
 
            if team_labels:
                last_label = max(team_labels, key=lambda t: t["top"])
                table_tops = [t.bbox[1] for t in tables]
                if not any(last_label["top"] <= top for top in table_tops):
                    pending_label = normalize_cell(last_label["name"])
 
    return pd.DataFrame(all_rows), pd.DataFrame(speaker_lookup)

def process_all_cards(tab_card):
    """
    Calls parse then adds Tournament, Season, and Year column while dropping Total. Combines all semester rows + speakers into one row
    """
    semester_rounds_df = [] 
    semester_speakers_df = [] 
 
    for pdf_path, tournament_name, season, year in tab_card:
        tournament_rounds_df, tournament_speaker_df = parse_tab_cards(pdf_path)
        
        if tournament_rounds_df.empty:
            continue
        tournament_rounds_df = tournament_rounds_df.drop(columns=["Total"], errors="ignore")
        tournament_rounds_df.insert(0, "Tournament", tournament_name)
        tournament_rounds_df.insert(1, "Season", season)
        tournament_rounds_df.insert(2, "Year", year)

        tournament_rounds_df = add_speaker_and_opponent_names(tournament_rounds_df, tournament_speaker_df)
        semester_rounds_df.append(tournament_rounds_df)
 
        tournament_speaker_df.insert(0, "Tournament", tournament_name)
        tournament_speaker_df.insert(1, "Season", season)
        tournament_speaker_df.insert(2, "Year", year)
        
        semester_speakers_df.append(tournament_speaker_df)
 
    return pd.concat(semester_rounds_df, ignore_index=True), pd.concat(semester_speakers_df, ignore_index=True)

Now we can parse the rounds

In [954]:
# Assigns a label for each semester of tab cards
semester_tab_paths =  [(FALL_2019_TAB_CARDS, "Fall_2019.csv"), 
                       (SPRING_2020_TAB_CARDS, "Spring_2020.csv"), (FALL_2020_TAB_CARDS, "Fall_2020.csv"),
                       (SPRING_2021_TAB_CARDS, "Spring_2021.csv"), (FALL_2021_TAB_CARDS, "Fall_2021.csv"),
                       (SPRING_2022_TAB_CARDS, "Spring_2022.csv"), (FALL_2022_TAB_CARDS, "Fall_2022.csv"),
                       (SPRING_2023_TAB_CARDS, "Spring_2023.csv"), (FALL_2023_TAB_CARDS, "Fall_2023.csv"),
                       (SPRING_2024_TAB_CARDS, "Spring_2024.csv"), (FALL_2024_TAB_CARDS, "Fall_2024.csv"),
                       (SPRING_2025_TAB_CARDS, "Spring_2025.csv"), (FALL_2025_TAB_CARDS, "Fall_2025.csv"),
                       (SPRING_2026_TAB_CARDS, "Spring_2026.csv")
                      ]
# Creates a df of the rounds and a speaker_df with all speakers seen
dfs = []
speakers = []
for (semester_card, label) in semester_tab_paths:
    df, speaker_df = process_all_cards(semester_card)

    # Cleans df by removing BYE rows or rows where speaker names are missing or ranks/speaks are zero
    df = df[
        ~df["G/O"].fillna("").str.strip().isin(["", "BYE"])
        & df["Speaker One Name"].fillna("").str.strip().ne("")
        & df["Speaker Two Name"].fillna("").str.strip().ne("")
        & df["Opponent One Name"].fillna("").str.strip().ne("")
        & df["Opponent Two Name"].fillna("").str.strip().ne("")
        & (df["Speaker One Rank"] != 0)
        & (df["Speaker Two Rank"] != 0)
        & (df["Opponent One Rank"] != 0)
        & (df["Opponent Two Rank"] != 0)
        & (df["Speaker One Speaks"] != 0.0)
        & (df["Speaker Two Speaks"] != 0.0)
        & (df["Opponent One Speaks"] != 0.0)
        & (df["Opponent Two Speaks"] != 0.0)
        & (df["W/L"] != "Wf")
        & (df["W/L"] != "Lf")
    ]
    df["Judge"] = df["Judge"].map(normalize_names)
    df['Judge'] = (
        df['Judge']
        .str.split(' - ').str[0]
        .str.replace(r'\s*\(V\)\s*', '', regex=True)
        .str.replace(r"\s*-\s*", "-", regex=True)
        .str.strip()
    )
    df["Speaker One Name"] = df["Speaker One Name"].str.replace(r"\s*-\s*", "-", regex=True)
    df["Speaker Two Name"] = df["Speaker Two Name"].str.replace(r"\s*-\s*", "-", regex=True)
    df["Opponent One Name"] = df["Opponent One Name"].str.replace(r"\s*-\s*", "-", regex=True)
    df["Opponent Two Name"] = df["Opponent Two Name"].str.replace(r"\s*-\s*", "-", regex=True)
    

    df.to_csv(label, index=False)
    dfs.append(df)
    speakers.append(speaker_df)

Combine and save the rounds

In [957]:
def collapse_mirrored_rounds(df):
    """
    Collapses mirrored round-pairs (same round logged once per team's
    perspective) into a single row per round, keeping the winning team's
    perspective. Falls back to keeping the row as-is if no clear winner
    is found (e.g. missing/unexpected W/L codes) or if the round was
    only logged once to begin with.
    """
    df = df.copy()
    df["W/L"] = df["W/L"].str.replace("Aw", "W")

    def team_key(a, b):
        return tuple(sorted([str(a).strip().lower(), str(b).strip().lower()]))

    df["_team1"] = df.apply(lambda r: team_key(r["Speaker One Name"], r["Speaker Two Name"]), axis=1)
    df["_team2"] = df.apply(lambda r: team_key(r["Opponent One Name"], r["Opponent Two Name"]), axis=1)
    df["_match_key"] = df.apply(lambda r: tuple(sorted([r["_team1"], r["_team2"]])), axis=1)
    df["_context_key"] = list(zip(df["Tournament"], df["Season"], df["Year"], df["Round"]))

    kept_rows = []

    for _, group in df.groupby(["_context_key", "_match_key"], sort=False):
        if len(group) == 1:
            kept_rows.append(group.iloc[0])
            continue

        winners = group[group["W/L"] == "W"]

        if len(winners) == 1:
            kept_rows.append(winners.iloc[0])
        else:
            kept_rows.append(group.iloc[0])

    result = pd.DataFrame(kept_rows).drop(columns=["_team1", "_team2", "_match_key", "_context_key"])
    return result.reset_index(drop=True)

In [958]:
rounds_dfs = pd.concat(dfs, ignore_index=True)
collapsed_rounds_dfs = collapse_mirrored_rounds(rounds_dfs)
collapsed_rounds_dfs.to_csv("Rounds.csv", index=False)

Now we need a list of debaters and the schools they are from. To accomplish this I will both extract the debate rosters directly from the APDA website, then I will analyize names of debaters with the Levenshtein distance algorithm in order to identify debaters and judges. Additionally, I compared the schools of previous partners of each name in order to construct a possible list of debaters. 

In [1399]:
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd

First, we need to create the paths, much like with the tab cards in order to parse the HTML pages to get our base roster.

In [1591]:
def get_debaters(school, year):
    """
    Creates file paths based on the contents of the data folder. 
    Uses a dictionary to replace file shortenings with the official APDA school name.
    Returns the file path.
    """
    paths = []
    for html in Path("Debaters").glob(f"{school}_{year}.html"):
        school = html.name.replace(f"_{year}.html", "")     
        school = {
            "BentleyUniversity": "Bentley University",
            "BGC": "Bard Graduate Center",
            "BinghamtonUniversity": "Binghamton University",
            "BostonCollege": "Boston College",
            "BostonUniversity": "Boston University",
            "BowdoinCollege": "Bowdoin College",
            "BPP": "Brierley Price Prior",
            "BrynMawr": "Bryn Mawr",
            "CarnegieMellon": "Carnegie Mellon",
            "CCSF": "City College of San Francisco",
            "ColumbiaLaw": "Columbia Law",
            "DuquesneUniversity": "Duquesne University",
            "FisherCollege":"Fisher College",
            "FIU": "Florida International University",
            "FloridaStateUniversity": "Florida State University",
            "FranklinandMarhall": "Franklin and Marshall",
            "GeorgeMason": "George Mason",
            "GeorgeWashington": "George Washington",
            "GroveCityCollege": "Grove City College",
            "HartHouse": "Hart House",
            "HarvardLaw": "Harvard Law",
            "HobartandWilliamSmith": "Hobart and William Smith",
            "IBADU": "University of Dhaka",
            "JohnsHopkins": "Johns Hopkins",
            "KwameNkrumahUniversityofScienceandTechnology": "Kwame Nkrumah University of Science and Technology",
            "LaVerne": "La Verne",
            "LoyolaMarymount": "Loyola Marymount",
            "LoyolaUniversityChicago": "Loyola University Chicago",
            "MoodyBibleInstitute": "Moody Bible Institute",
            "MorehouseCollege": "Morehouse College",
            "MountHolyoke": "Mount Holyoke",
            "NotreDame": "Notre Dame",
            "PatrickHenry": "Patrick Henry",
            "PrinceGeorgesCommunityCollege": "Prince George's Community College",
            "ProvidenceCollege": "Providence College",
            "QueensUniversity": "Queen's University",
            "SanJoseState": "San Jose State",
            "SantaClara":"Santa Clara",
            "SimonFraserUniversity": "Simon Fraser University",
            "SimonsRockCollege": "Simon's Rock College",
            "SouthCarolina": "South Carolina",
            "StAndrews": "St. Andrews",
            "StJohns": "St. Johns",
            "StMarys": "St. Mary's",
            "StonyBrookUniversity": "Stony Brook University",
            "TelAviv": "Tal Aviv",
            "TheCollegeofNewJersey": "The College of New Jersey",
            "UCDL&H": "UCD L&H",
            "UniversityofAlaskaAnchorage": "University of Alaska Anchorage",
            "UniversityofAlbany": "University of Albany",
            "UniversityofBritishColumbia": "University of British Columbia",
            "UniversityofCalgary": "University of Calgary",
            "UniversityofChicago": "University of Chicago",
            "UniversityofDelaware": "University of Delaware",
            "UniversityofDenver": "University of Denver",
            "UniversityofGuelph": "University of Guelph",
            "UniversityofHawaiiatManoa": "University of Hawaii at Manoa",
            "UniversityofMassachusetts": "University of Massachusetts",
            "UniversityofMichigan": "University of Michigan",
            "UniversityofMinnesota": "University of Minnesota",
            "UniversityofNewSouthWales": "University of New South Wales",
            "UniversityofNorthCarolina": "University of North Carolina",
            "UniversityofPittsburgh": "University of Pittsburgh",
            "UniversityofSouthernCalifornia": "University of Southern California",
            "UniversityofSydney": "University of Sydney",
            "UniversityofthePeople": "University of the People",
            "UniversityofVermont": "University of Vermont",
            "UniversityofVirginia": "University of Virginia",
            "UniversityofWaterloo": "University of Waterloo",
            "UTAustin": "UT Austin",
            "WashingtonUniversityinStLouis": "Washington University in St. Louis",
            "WilfredLaurierUniversity": "Wilfred Laurier University",
            "WilliamandMary": "William and Mary",
            "YorkUniversity": "York University",
        }.get(school, school)
        
        paths.append((html, school, year))
    return paths

Then we may extract the rosters for each school

In [1592]:
# Assigns a label for each school's roster
schools = [("American", "AMERICAN"), ("Adelphi", "ADELPHI"), ("Amherst", "AMHERST"), ("Bard", "BARD"), 
           ("Bates", "Bates"), ("BentleyUniversity", "BENTLEYUNIVERSITY"), ("Berkeley", "BERKELEY"), 
           ("BGC", "BGC"), ("BinghamtonUniversity", "BINGHAMTONUNIVERSITY"), ("BostonCollege", "BOSTONCOLLEGE"), 
           ("BostonUniversity", "BOSTONUNIVERSITY"), ("BowdoinCollege", "BOWDOINCOLLEGE"), ("BPP", "BPP"), 
           ("Bradley", "BRADLEY"), ("Brandeis", "BRANDEIS"), ("Brown", "BROWN"), ("BrynMawr", "BRYNMAWR"), ("Bucknell", "BUCKNELL"),
           ("Cambridge", "CAMBRIDGE"), ("Carleton", "CARLETON"), ("CarnegieMellon", "Carnegie Mellon"), ("CCSF", "CCSF"),
           ("Claremont", "CLAREMONT"), ("Colgate", "COLGATE"), ("Columbia", "COLUMBIA"), ("ColumbiaLaw", "COLUMBIALAW"),
           ("Cornell", "CORNELL"), ("CUNY", "CUNY"), ("Dalhousie", "DALHOUSIE"), ("Dartmouth", "DARTMOUTH"), ("Davidson", "DAVIDSON"),
           ("Denison", "DENISON"), ("Drexel", "DREXEL"), ("Duke", "DUKE"), ("DuquesneUniversity", "DUQUESNEUNIVERSITY"), 
           ("Durham", "DURHAM"), ("Emmanuel", "EMMANUEL"), ("Emory", "EMORY"), ("Fairfield", "FAIRFIELD"), ("FisherCollege", "FISHERCOLLEGE"), 
           ("FIU", "FIU"), ("FloridaStateUniversity", "FLORIDASTATEUNIVERSITY"), ("Fordham", "FORDHAM"), 
           ("FranklinandMarshall", "FRANKLINANDMARSHALL"), ("GeorgeMason", "GEORGEMASON"), ("Georgetown", "GEORGETOWN"), 
           ("GeorgeWashington", "GEORGEWASHINGTON"), ("Glasgow", "GLASGOW"), ("Grinnell", "GRINELL"), 
           ("GroveCityCollege", "GROVECITYCOLLEGE"), ("Hamilton", "HAMILTON"), ("HartHouse", "HARTHOUSE"), ("Harvard", "HARVARD"), 
           ("HarvardLaw", "HARVARDLAW"), ("Haverford", "HAVERFORD"), ("HWS", "HWS"), ("HobartandWilliamSmith", "HOBARTANDWILLIAMSMITH"),
           ("IBADU", "IBADU"), ("IIUM", "IIUM"), ("JohnsHopkins", "JOHNSHOPKINS"), ("Kings", "KINGS"), 
           ("KwameNkrumahUniversityofScienceandTechnology", "KWAMENKRUMAHUNIVERSITYOFSCIENCEANDTECHNOLOGY"), ("LaVerne", "LAVERNE"), 
           ("Lehigh", "LEHIGH"), ("LoyolaMarymount", "LOYOLAMARYMOUNT"), ("LoyolaUniversityChicago", "LOYOLAUNIVERSITYCHICAGO"), 
           ("Maryland", "MARYLAND"), ("McGill", "MCGILL"), ("Middlebury", "MIDDLEBURY"), ("MIT", "MIT"),
           ("MoodyBibleInstitute", "MOODYBIBLEINSTITUTE"), ("Morehouse", "MOREHOUSE"), ("MorehouseCollege", "MOREHOUSECOLLEGE"),
           ("MountHolyoke", "MOUNTHOLYOKE"), ("Northeastern", "NORTHEASTERN"), ("Northwestern", "NORTHWESTERN"), 
           ("NotreDame", "NOTREDAME"), ("NYU", "NYU"), ("Odette", "ODETTE"), ("Ottawa", "OTTAWA"), ("Oxford", "OXFORD"), 
           ("Pace", "PACE"), ("PatrickHenry", "PATRICKHENRY"), ("Penn", "PENN"), 
           ("PrinceGeorgesCommunityCollege", "PRINCEGEORGESCOMMUNITYCOLLEGE"), ("Princeton", "PRINCETON"), 
           ("ProvidenceCollege", "PROVIDENCECOLLEGE"), ("Quakers", "QUAKERS"), ("QueensUniversity", "QUEENSUNIVERSITY"), ("RIT", "RIT"),
           ("Rochester", "ROCHESTER"), ("RPI", "RPI"), ("Rutgers", "RUTGERS"), ("SanJoseState", "SANJOSESTATE"), ("SantaClara", "SANTACLARA"),
           ("SimonFraserUniversity", "SIMONFRASERUNIVERSITY"), ("SimonsRockCollege", "SIMONSROCKCOLLEGE"), ("Skidmore", "SKIDMORE"), 
           ("Smith", "SMITH"), ("SouthCarolina", "SOUTHCAROLINA"), ("Spelman", "SPELMAN"), ("StAndrews", "STANDREWS"), ("Stanford", "STANFORD"), 
           ("StJohns", "STJOHNS"), ("StMarys", "STMARYS"), ("StonyBrookUniversity", "STONYBROOKUNIVERSITY"), ("Swarthmore", "SWARTHMORE"),
           ("Sydney", "SYDNEY"), ("Syracuse", "SYRACUSE"), ("TelAviv", "TELAVIV"), ("Temple", "TEMPLE"), ("TESU", "TESU"),
           ("TheCollegeofNewJersey", "THECOLLEGEOFNEWJERSEY"), ("Trinity", "TRINITY"), ("Tufts", "TUFTS"), ("Tulane", "TULANE"), 
           ("Tulsa", "TULSA"), ("UCDL&H", "UCDL&H"), ("UCLA", "UCLA"), ("UConn", "UCONN"), ("UMBC", "UMBC"), ("UniversityofAlaskaAnchorage", "UNIVERSITYOFALASKAANCHORAGE"),
           ("UniversityofAlbany", "UNIVERSITYOFALBANY"), ("UniversityofBritishColumbia", "UNIVERSITYOFBRITISHCOLUMBIA"), 
           ("UniversityofCalgary", "UNIVERSITYOFCALGARY"), ("UniversityofChicago", "UNIVERSITYOFCHICAGO"), ("UniversityofDelaware", "UNIVERSITYOFDELAWARE"),
           ("UniversityofDenver", "UNIVERSITYOFDENVER"), ("UniversityofGuelph", "UNIVERSITYOFGUELPH"), ("UniversityofHawaiiatManoa", "UNIVERSITYOFHAWAIIATMANOA"),
           ("UniversityofMassachusetts", "UNIVERSITYOFMASSACHUSETTS"), ("UniversityofMichigan", "UNIVERSITYOFMICHIGAN"), ("UniversityofMinnesota", "UNIVERSITYOFMINNESOTA"),
           ("UniversityofNewSouthWales", "UNIVERSITYOFNEWSOUTHWALES"), ("UniversityofNorthCarolina", "UNIVERSITYOFNORTHCAROLINA"), 
           ("UniversityofPittsburgh", "UNIVERSITYOFPITTSBURGH"), ("UniversityofSouthernCalifornia", "UNIVERSITYOFSOUTHERNCALIFORNIA"),
           ("UniversityofSydney", "UNIVERSITYOFSYDNEY"), ("UniversityofthePeople", "UNIVERSITYOFTHEPEOPLE"), ("UniversityofVermont", "UNIVERSITYOFVERMONT"),
           ("UniversityofVirginia", "UNIVERSITYOFVIRGINIA"), ("UniversityofWaterloo", "UNIVERSITYOFWATERLOO"), ("UTAustin", "UTAUSTIN"), ("Vassar", "VASSAR"),
           ("Villanova", "VILLANOVA"), ("WashingtonUniversityinStLouis", "WASHINGTONUNIVERSITYINSTLOUIS"), ("Wellesley", "WELLESLEY"), ("Wesleyan", "WESLEYAN"),
           ("Western", "WESTERN"), ("WilfredLaurierUniversity", "WILFREDLAURIERUNIVERSITY"), ("WilliamandMary", "WILLIAMANDMARY"), 
           ("Williams", "WILLIAMS"), ("WSCC", "WSCC"), ("YorkUniversity", "YORKUNIVERSITY"), ("Yale", "YALE")]

# Creates a df with all speakers seen in each school
all_debaters = []
for school, label in schools:
    school_debaters = []
    for year in range(2004, 2026):
        for html_path, school_name, year in get_debaters(school, year):

            with open(html_path, encoding="utf-8") as fp:
                soup = BeautifulSoup(fp, "html.parser")

            target_table = None

            for table in soup.find_all("table"):
                headers = [th.get_text(strip=True) for th in table.find_all("th")]
                if headers == ["ID", "Name", "Year on Team"]:
                    target_table = table
                    break

            if target_table is None:
                continue

            df = pd.read_html(StringIO(str(target_table)))[0]
            df["School"] = school_name
            df["Initial Year"] = year + 1 - df["Year on Team"]
            df["Last Year Seen"] = year
            df = df.drop(columns=["ID", "Year on Team"])
            school_debaters.append(df)
    
    if not school_debaters:
        continue
    df = pd.concat(school_debaters, ignore_index=True)    
    df = (
        df.groupby(["Name", "School"], as_index=False)
          .agg({
              "Initial Year": "min",
              "Last Year Seen": "max"
          })
    )
    df = df.drop_duplicates(keep="last")
    all_debaters.append(df)
    
    df.to_csv(f"DebateRosters/{school}_debaters.csv", index=False)

all_debaters = pd.concat(all_debaters, ignore_index=True)
all_debaters = all_debaters.rename(columns={"Name": "Debater"})
all_debaters = all_debaters.drop_duplicates(subset=["Debater"], keep="last")
all_debaters.to_csv("All_Debaters.csv", index=False)

I manually used the following Levenshtein distances in order to view the top 10 options and make manual adjustments.

In [1548]:
import Levenshtein
import pandas as pd

judge_roster = pd.read_csv("All_debaters.csv") 
judge_names = judge_roster["Debater"].tolist()

TOP_N = 10

results = []

for lost_name, rounds_lost in indiv_judges.items():
    scored = [(name, Levenshtein.distance(lost_name, name)) for name in judge_names]
    scored.sort(key=lambda x: x[1])

    top_matches = scored[:TOP_N]

    for rank, (name, dist) in enumerate(top_matches, start=1):
        results.append({
            "Lost Judge": lost_name,
            "Rounds Lost": rounds_lost,
            "Rank": rank,
            "Candidate Match": name,
            "Distance": dist,
        })

judge_match_suggestions = pd.DataFrame(results)
judge_match_suggestions.to_csv("lost_judge_top_matches.csv", index=False)

In [1549]:
import Levenshtein

debate_roster = pd.read_csv("All_Debaters.csv")
roster_names = debate_roster["Debater"].tolist()

TOP_N = 10 

results = []

for _, row in lost_rounds_all.iterrows():
    lost_name = row["Name"]

    scored = [(roster_name, Levenshtein.distance(lost_name, roster_name))
              for roster_name in roster_names]
    scored.sort(key=lambda x: x[1])

    top_matches = scored[:TOP_N]

    for rank, (roster_name, dist) in enumerate(top_matches, start=1):
        results.append({
            "Lost Name": lost_name,
            "Role": row["Role"],
            "Rounds Lost": row["Rounds Lost"],
            "Rank": rank,
            "Candidate Match": roster_name,
            "Distance": dist,
        })

match_suggestions = pd.DataFrame(results)
match_suggestions.to_csv("lost_name_top_matches.csv", index=False)

The dictionary that follows makes adjustments based on the Levenshtein distances.

In [1582]:
COMMON_MISSPELLINGS = {
    "Jenni Pham": "Jenny Pham",
    "Jenni": "Jenny Pham",
    "Dominic Deramo": "Dominic DeRamo",
    "Joseph Rubas": "Julia Rubas",
    "Joey Rubas": "Julia Rubas",
    "Audrey J. Higley": "Audrey Higley",
    "Alessandro Perri": "Ale Perri",
    "Cece Szkutak": "CeCe Szkutak",
    "Nicholas Devito": "Nick DeVito",
    "Nick Devito": "Nick DeVito",
    "Kj Kniering": "KJ Kniering",
    "Ian Mcvann-Henkelmann": "Ian McVann-Henkelmann",
    "Ian Mcvann": "Ian McVann-Henkelmann",
    "Ian Mcvann-Henk Elmann": "Ian McVann-Henkelmann",
    "Ian Mcvann-Henke Lmann": "Ian McVann-Henkelmann",
    "Ian Mcvann-Henklemann": "Ian McVann-Henkelmann",
    "Ian Mcvann Henkelmann": "Ian McVann-Henkelmann",
    "Ian Mcvann Henkelma Nn": "Ian McVann-Henkelmann",
    "Izzy Jenkins": "Isabella Jenkins",
    "Max F. Neuman": "Max Neuman",
    "Max Neumann": "Max Neuman",
    "Naomi Mckenna": "Naomi McKenna",
    "Alejandro Franqui-Ferrer": "Alejandro Franqui",
    "Katherine St George": "Katherine St. George",
    "Claire Mcmahon Fishman": "Claire Fishman",
    "Eli Nelson": "Elijah Nelson",
    "Hannah Platter They/Them": "Hannah Platter",
    "Oliver Mccammon": "Oliver McCammon",
    "Eva Bruce She/Her": "Eva Bruce",
    "Andrew Kao*": "Andrew Kao",
    "Adam T Harrington": "Adam Harrington",
    "Ej Hermacinski": "EJ Hermacinski",
    "Ceci Granda-Scott": "Cecilia Granda-Scott",
    "Cam Chacon": "Cameron Chacon",
    "Sam Watkins": "Samuel Watkins",
    "Liz Esterbrook": "Elizabeth Esterbrook",
    "Katie Farrell": "Kathryn Farrell",
    "Wes Mcgovern": "Wes McGovern",
    "Aadhav Raviarasan": "Aadhavaarasan Raviarasan",
    "Yashas Mallikarju N": "Yashas Mallikarjun",
    "Gordon Mcneill": "Gordon McNeill",
    "Drew Harrington": "Andrew Harrington",
    "Alice Marchant": "Alice Merchant",
    "Vincent Kazella": "Vinny Kazz",
    "Vinny Kazella": "Vinny Kazz",
    "Muzzi Godil": "Muzamil Godil",
    "Muzzi": "Muzamil Godil",
    "Stav Kanza": "Stav Kanza-Levi",
    "Aleisha Martinez Sandoval": "Aleisha Martínez",
    "Aleisha Martinez-Sandoval": "Aleisha Martínez",
    "Samhitha Duggirala": "Samhitha Duggurala",
    "Sheryar Fazal": "Sheryar Ahmad Fazal",
    "Sam Widell": "Sam Widwell",
    "Saif Elkhodor": "Saif El Khodor",
    "Adi Jayakrishnan":"Aditya Jayakrishnan",
    "Madeleine Watson": "Maddie Watson",
    "Giuseppe Dimassa": "Giuseppe DiMassa",
    "Benjamin Grimes": "Benji Grimes",
    "Cassie Fitts": "Cassandra Fitts",
    "Abby Fechisso": "Abigail Fechisso",
    "Madeline Cheshire": "Maddy Cheshire",
    "Tamhid Islam": "Tahmid Islam",
    "Janul De Silva": "Janul de Silva",
    "Claire Fraise": "Claire Frase",
    "Allison Ross": "Ally Ross",
    "Zach Lemonides": "Zachary Lemonides",
    "Alexander Gerber": "Alex Gerber",
    "Gabe Lomonaco": "Gabriel Lomonaco",
    "Aidan Hollinger Miles": "Aidan Hollinger-Miles",
    "Zachary Malek": "Zack Malek",
    "Theo Miranda-Zellnik": "Theo Miranda-Zellink",
    "Becca Shields": "Rebecca Shields",
    "Gabriel Ritter": "Gabe Ritter",
    "Gabe Lomonaco": "Gabriel Lomonaco", 
    "Gabe Lomanaco": "Gabriel Lomonaco",
    "Eva Quinones": "Eva-Mare Quinones",
    "Will Arnesen": "William Arnesen",
    "Da'Von Boyd": "Davon Boyd", 
    "Anne Motovillof": "Anne Motoviloff",
    "Nikos Efthymiadis": "Nikolaos Efthymiadis",
    "Will Zeng": "William Zeng",
    "Nokutenda Zuze": "Noku Zuze",
    "Joanne Bai": "Joanna Bai",
    "Dillion Ma": "Dillon Ma",
    "Omesh": "Omesh Dhar Dwivedi",
    "Andrew Bellows": "Andrew Bell",
    "Nat Puapattanakajorn": "Nat Puappatankajorn",
    "Matt Feng": "Matthew Feng",
    "Tosca Neuma Nn": "Tosca Neumann",
    "Bartholomew Kaminski": "Bart Kaminski",
    "Dylan Gyauch-Lewis": "Dylan Gyauch-Lewis",
    "Benjamin Scherzer": "Ben Scherzer",
    "Brendan Mcdermott": "Brendan Mcdermott",
    "Alexander Purn": "Alex Purn",
    "Eddie Siderenko": "Eddie Sidoxrenko",
    "Katie Zhang": "Kate Zhang",
    "Peter Kladais": "Peter Kladias",
    "Jaiden Hassan": "Jaiden Hasan",
    "Muku Madzivire": "Mukudzeiishe Madzivire",
    "Pranav Gargipati": "Pranav Garigipati",
    "Sonam Wangchuk": "Sonam Tenzin Wangchuk",
    "Max Sheremeta": "Maxwell Sheremeta",
    "Aditya Ram": "Adi Ram",
    "Robert Neilson": "Robert Nielsen",
    "Leandro Guevara": "Leandro Guevara-Neyra",
    "Timothy Goggin": "Tim Goggin",
    "Matthew Rubenstein": "Matt Rubenstein",
    "Tori Reiz": "Tori Reisz",
    "Devyani :": "Devyani Goel",
    "Bella Sorial": "Isabella Sorial",
    "Ale": "Ale Perri",
    "Aiza": "Aiza Nygman",
    "Ari Hanh": "Ari Hahn",
    "Maxwell Sheremata": "Maxwell Sheremeta",
    "Harry Carr": "Harrison Carr",
    "Samantha Wing": "Samantha Wong",
    "Will Donnely": "Will Donnelly",
    "Shruti Narayanabha Tla": "Shruti Narayanabhatla",
    "Alexa Ross": "Ally Ross",
    "Sami Cuaresma": "Samara Cuaresma",
    "Xiao-Ke Lu": "Xiao-ke Lu",
    "Nikki Schuldt": "Niki Schrift",
    "Christopher O’Keeffe":	"Christopher O'Keeffe",
    "Brendan Mcdermott": "Brendan McDermott",
    "Nicholas Lim": "Nicolas Lim",
    "Joon Sohn": "Joonpyo Sohn",
    "Dylan Gyauch-Lewis": "Dylan Gyauch Lewis",
    "Alexandra Dischler": "Alex Dischler",
    "Siddharth Ramanathan": "Sid Ramanathan",
    "Matt Rohn": "Matthew Rohn",
    "Ina Ralhakar": "Ina Rahalkar",
    "Ong Unjiwatana": "Ong Unjitwattana",
    "Rob Nielsen": "Robert Nielsen",
    "Cornelia": "Cornelia Hsieh",
    "Mitch Mullen": "Mitchell Mullen",
    "Kaley Kathleen Alexandre-Burke": "Kaley Alexandre-Burke",
    "Angier Li": "Angier Lei",
    "Gabbi Schilcusky": "Gabbi Shilcusky",
    "Mehul Agrawal": "Mehul Agarwal",
    "Mindy Hupsen": "Mindy Huspen",
    "Andrew Montieth": "Andrew Monteith",
    "Fee Pelz-Sharpe": "Fiona Petz-Sharpe",
    "Preston Johnston": "Preston Johnson",
    "Sonam": "Sonam Tenzin Wangchuk",
    "Andrew Liang": "Andrew Laing",
    "Reca Safarti": "Reca Sarfati",
    "Will Meyer": "William Meyer",
    "Anthony Peña": "Anthony Pena",
    "Rishven K Pravin": "Rishven Pravin",
    "Gregory Gentile": "Greg Gentile",
    "Matt Simons": "Matthew Simons",
    "Addie Gill T": "Addie Gill",
    "Micahel Ryter": "Michael Ryter",
    "Martin Gazsner": "Martin Gaszner",
    "Ari Han": "Ari Hahn",
    "Nick Cathcart": "Nicolas Cathcart",
    "Andew Monteith": "Andrew Monteith",
    "Liberty": "Liberty Prieb",
    "Cody Mcmanus": "Cody McManus",
    "Foula Christopoulos": "Foula Christopolous",
    "Michelle Teicher": "Michelle Tiecher",
    "Hannah Owens Pierre": "Hannah Owens-Pierre",
    "William Howard-Driemeier": "William Hallward-Driemeier",
    "Ej Kang": "EJ Kang",
    "William Huang": "Will Huang",
    "Kyuryeon Kim": "Khuryeon Kim",
    "Pètra De Beer": "Petra de Beer",
    "Erica Morelli": "Erica Morellli",
    "Arushi Agrawal": "Arushi Agarwal",
    "Uszee Mckoy": "UsZee Mckoy",
    "Efrain Thomas Ortiz": "Efrain Ortiz",
    "Vivienne Montiero": "Vivienne Monteiro",
    "Benjamin Cortez": "Ben Cortez",
    "Yva": "Yva Totchum",
    "Elena Blake Leeds": "Elena Leeds",
    "Tammy Yamile Leon Molina": "Tammy Yamile León Molina",
    "Sophia Tyrrell Knott": "Sophia Tyrrell-Knott",
    "Rennie": "Rennie Lee",
    "Sumanth M": "Sumanth Mahalingam",
    "Jaice": "Jaice Williamson",
    "Josh Sampson": "Joshua Sampson",
    "Jeffery Gao": "Jeffrey Gao",
    "Andres Mendoza Casas": "Andres Mendozas Casas",
    "Matt Lee": "Mathew Lee",
    "Dan Perez": "Daniel Perez",
    "Claire Mchahon Fishman": "Claire Fishman",
    "Cecilia Szkutak": "CeCe Szkutak",
    "Jela Shriver": "Jela Shiver",
    "Matthew Franco": "Matt Franco",
    "Max Kornfield": "Max Kornfeld",
    "Egor Cherniuk": "Egor Chernyuk",
    "Catie Macauley": "Catie Mccauley",
    "Will Choi": "William Choi",
    "Carlos Irrisari": "Carlos Irisarri",
    "Charlie Mclarnon": "Charlie McLarnon",
    "Dilay Kalinoglu": "Dilay Kolinoglu",
    "Clemente Nicado-Yelmene": "Clemente Nicado Yelmene",
    "Natalie Keim": "Nat Keim",
    "Joseph Mcgroarty": "Joseph McGroarty",
    "Ananya Ganish": "Ananya Ganesh",
    "Sumanth Mahalinga M": "Sumanth Mahalingam",
    "Marcel": "Marcel Cato",
    "Arielle Gallagos": "Arielle Gallegos",
    "Khy Stubblefield": "Khylan Stubblefield",
    "Zander": "Zander Jeinthanuttkanont",
    "Neftalí Reynoso": "Neftali Reynoso",
    "Sheryar": "Sheryar Ahmad Fazal",
    "Teddy Jack": "TJ Jack",
    "Rafael Rodriguez": "Rafael Rodriguez Alvarez",
    "Will Logue": "William Logue",
    "Adi Jayakrishna N": "Aditya Jayakrishnan",
    "Gaurav": "Gaurav Gawankar",
    "Joe Brennan": "Joeseph Brennan",
    "Alina Ziying Su": "Alina Su",
    "Jiwhan Moon": "JiWhan Moon",
    "Jack Reeed": "Jack Reed",
    "Arianna Hellman T": "Arianna Hellman",
    "Paola Apolinari O": "Paola Apolinario",
    "Lizzie Walters": "Lizzie Waters",
    "Julia Shepard": "Julia Shephard",
    "Dominic !": "Dominic DeRamo",
    "Gian Luigi Zaninelli": "GianLuigi Zaninelli",
    "Vara Mathiylaakan": "Vara Mathiyalakan",
    "Nat Nichanun Puapattanakajorn": "Nat Puappatankajorn",
    "Sid Chakravarthy": "Siddhaarth Chakravarthy",
    "Susan Mcharris": "Susan McHarris",
    "Sydney C": "Sydney Cook",
    "Sarah Cobau-": "Sarah Cobau",
    "Ryan Geary .": "Ryan Geary",
    "Spike": "Spike King",
    "Andrew Harrington .": "Andrew Harrington",
    "Ry-Ry Geary": "Ryan Geary",
    "Alex Uy-Tioco": "Alexandra Uy-Tioco",
    "James Cox-Donovan": "James Donovan",
    "Mikala Parnell": "Mikala Pernell",
    "Rishika Deshide": "Rish Deshide",
    "Violet Whitimire": "Violet Whitmire",
    "Alejandro Franqui Ferrer": "Alejandro Franqui",
    "Eva Marie Quinones": "Eva Marie-Quinones",
    "Sania Ifran": "Sania Irfan",
    "Riya Singh": "Rhea Singh",
    "Lindsey Gradow Ski": "Lindsey Gradowski",
    "Rishve N Pravin": "Rishven Pravin",
    "Pranav Garigip Ati": "Pranav Garigipati",
    "Ong Unjiwata Na": "Ong Unjitwattana",
    "William Hallward-Dri Emeier": "William Hallward-Driemeier",
    "Oscar Cloutier": "Oscar Cloutier Potter",
    "Oscar Clouti Er Potter": "Oscar Cloutier Potter",
    "Domi Nic Dera Mo": "Dominic DeRamo",
    "Eftychia  Christodoulou": "Eftychia Christodoulou",
    "Melina Piatta-Chayan": "Melina Piatti-Chayan",
    "Sophia Winner-": "Sophia Winner",
    "Dylan Gyauch-Lewis": "Dylan Gyauch Lewis",
    "Giuseppe Di Massa": "Giuseppe DiMassa",
    "Gianluigi Zaninelli": "GianLuigi Zaninelli",
    "Kathleen Mcintyre": "Kathleen McIntyre",
    "Ana Carolina Marques Perez": "Ana Marques Perez",
    "Kaya Panchalinga M": "Kaya Panchalingam",
    "Aadhav Raviarasan": "Aadhavaarasan Raviarasan",
    "Aadhavaarasa N Raviarasan": "Aadhavaarasan Raviarasan",
    "Keshav Malik Kapoor": "Keshav Kapoor",
    "Abby Hill": "Abigail Hill",
    "Ceci Granda Scott": "Cecilia Granda-Scott",
    "Cecilia Granda Scott": "Cecilia Granda-Scott",
    "Emma Listgarte N": "Emma Listgarten",
    "Zander Jeinthanuttkano Nt": "Zander Jeinthanuttkanont",
    "Zander Jeinthannatkanont": "Zander Jeinthanuttkanont",
    "Abdullah Mejjalid": "Abdullah Mejjallid",
    "Abhishek Amit Shah": "Abhishek Shah",
    "Aidan Duran Rey": "Adrian Duran",
    "Aidan Gilles": "Aidan Gillies",
    "Aiden Shannon": "Aidan Shannon",
    "Aislinn O'Brian": "Aislinn O'Brien",
    "Akash": "Akash Shivakumaar",
    "Alexander Gordon": "Alex Gordon",
    "Alexander Schramm": "Alex Schramm",
    "Alexandra Uy-Tico": "Alexandra Uy-Tioco",
    "Allison Chan": "Alison Chan",
    "Alvaro Marin-Garcia": "Alvaro Marin Garcia",
    "Amala Kari": "Amala Karri",
    "Amira Butani": "Amira Bhutani",
    "An Lahn Le": "An-Lanh Le",
    "An Lanh Le": "An-Lanh Le",
    "Ana Marquez Perez": "Ana Marques Perez",
    "Andrew Harington": "Andrew Harrington",
    "Andrew Rozenbilt": "Andrew Rozenblit",
    "Anna Keternos": "Anna Ketrenos",
    "Ariana Hellman": "Arianna Hellman",
    "Audrey J Higley": "Audrey Higley",
    "Audri Bhowmick": "Audri Bhomick",
    "Ava Schneiburg": "Ava Schneiberg",
    "Awsam Boaubid":"Awsam Bouabid",
    "Zach Braunstein":"Zachary Braunstein",
    "Alex Elsrodt": "Alex Elstrodt",
    "Elena Lille": "Elena Lill",
    "Zimo Tracy Ge": "Zimo-Tracy Ge",
    "Zhouai Joann Yu": "Zhouai Joann",
    "Zan Rosen": "Zan Rozen",
    "William Shachar": "Will Shachar",
    "Tori Fekete": "Victoria Fekete",
    "Clemente Yelmene Nicado": "Clemente Nicado Yelmene",
    "Alex Elsdrodt": "Alex Elstrodt",
    "Sophia Mason": "Sophie Mason",
    "Zara Mermon": "Zara Memon",
    "Zachary Brown": "Zach Brown",
    "Yuqiu Rachel Liu": "Yuqian Li",
    "Yanni Trimiklionitis": "Yanni Trimikliniotis",
    "Yana Sharifulli Na": "Yana Sharifullina",
    "Xiao Ke Lu": "Xiao-ke Lu",
    "Wilson Shen": "Wilson Chen",
    "Wilson Cheung": "Winson Cheung",
    "William-Hallward-Driemeier": "William Hallward-Driemeier",
    "William Hallward-Dreiemeir": "William Hallward-Driemeier",
    "William Hallward Driemeier": "William Hallward-Driemeier",
    "Vita Raskevičiūtė": "Vita Raskeviciute",
    "Viraj": "Viraj Nautiyal",
    "Vikram Balasubramania N": "Vikram Balasubramanian",
    "Vihini Gunaseker A": "Vihini Gunasekera",
    "Vignesh Mehrotra": "Vighnesh Mehrotra",
    "Viet Thé Phan": "Viet Phan",
    "Vasily Syomin": "Vasiley Syomin",
    "Vasa Syomin": "Vasiley Syomin",
    "Varsha Sripadha M": "Varsha Sripadham",
    "Trey Garcia Schartz": "Trey Garcia-Schartz",
    "Tilly Swanson": "Matilda Swanson",
    "Thomas Ii Hyun Kim": "Thomas Kim",
    "Theodore Gerken": "Theodore Gercken",
    "Theodore Gercken2": "Theodore Gercken",
    "Theodore": "Theodore Gercken",
    "Theodor Gerken": "Theodore Gercken",
    "Tanya": "Tanya Chatterjee",
    "Tai Hendricks": "Tai Henrichs",
    "Pranav Garigapati": "Pranav Garigipati",
    "Pranav Garigiapti": "Pranav Garigipati",
    "Pranav Gpt": "Pranav Garigipati",
    "Oscar Cloutier-Potter": "Oscar Cloutier Potter",
    "Oscar Potter": "Oscar Cloutier Potter",
    "Szilveszter Palvolgyi": "Szilvester Palvolgyi",
    "Ollie Saunder S": "Olivia Saunders",
    "Ollie Saunders": "Olivia Saunders",
    "Nicole Salinas Reyes": "Nicole Salinas-Reyes",
    "Nichanun Puapattanakajom": "Nat Puappatankajorn",
    "Nat Puapattankajor N": "Nat Puappatankajorn",
    "Muzammil Godil": "Muzamil Godil",
    "Muzzamil Godil": "Muzamil Godil",
    "Maxwell Hurowitz": "Max Hurowitz",
    "Sankey Bhalotia": "Sanket Bhalotia",
    "Sanket Bhlatoia": "Sanket Bhalotia",
    "Santiago Cantu": "Santi Cantu",
    "Sara Zdancewi Cz": "Sara Zdancewicz",
    "Susanna Kirtzler": "Susanna Kritzler",
    "Sulley Ho": "Sullivan Ho",
    "Srishti Ghosh": "Srishti Gosh",
    "Sophie Rose Fetter": "Sophie Fetter",
    "Sophia Torres Da Cruz": "Sophia Torres da Cruz",
    "Siddharth Ramanatha N": "Sid Ramanathan",
    "Shruthi Bharath Kumar": "Sruthi Bharath Kumar",
    "Siddharth Ramanathan": "Sid Ramanathan",
    "Siddharth Ramanathan": "Sid Ramanathan",
    "Scott Santella": "Scott Santaella",
    "Sav Stackhouse": "Savannah Stackhouse",
    "Samuel Arneson": "Sam Arnesen",
    "Samantha Pryzbisiki": "Samantha Przybisiki",
    "Sam Sulzinksy": "Sam Sulzinsky",
    "Sam Melcher": "Samuel Melcher",
    "Sajan Mehrota": "Sajan Mehrotra",
    "Saif El-Khodor": "Saif El Khodor",
    "Ryyaan Sheikh": "Rayan Sheikh",
    "Ryan Tiedmann": "Ryan Tiedemann",
    "Ryan Tiedema Nn": "Ryan Tiedemann",
    "Ryan G": "Ryan Tiedemann",
    "Romina Lillolari": "Romina Lilollari",
    "Rocco Spinozzi-D’Andrea": "Rocco Spinozzi-D'Andrea",
    "Robert Redine": "Robert Rendine",
    "Robert Neilsen": "Robert Nielsen",
    "Rita Mayevsyaka": "Rita Mayevskaya",
    "Rish Pravin": "Rishven Pravin",
    "Ricky Kiamliev": "Ricky Kiamilev",
    "Reya Kalolwala": "Reyya Kalolwala",
    "Rachel Robbin": "Rachel Robin",
    "Pranav Sasikumae": "Pranav Sasikumar",
    "Pragyna Yerramalli": "Pragnya Yerramalli",
    "Petra De Beer": "Petra de Beer",
    "Peregrine Beckett": "Perry Beckett",
    "Patrik Dugan": "Patrick Dugan",
    "Patrick Mccarthy": "Patrick McCarthy",
    "Thibaut Juneja": "Shanyu Thibaut Juneja",
    "Sam Rowher": "Sam Rohwer",
    "Sam Duggirala": "Samhitha Duggurala",
    "Sam Arneson": "Sam Arnesen",
    "Rocco Spinozzi D'Andrea": "Rocco Spinozzi-D'Andrea",
    "Nitin Kumar Saidha": "Nitin Kumar",
    "Nikita Chakraborty": "Nikhita Chakraborty",
    "Nicole Kagain": "Nicole Kagan",
    "Nicolas Parra": "Nick Parra",
    "Marcelo Parra" : "Marcelo Rodriguez Parra",
    "Nico Hortiguera": "Nicolas Hortiguera",
    "Nicholas Mccarthy": "Nicholas McCarthy",
    "Nichanun Puapattanakajorn": "Nat Puappatankajorn",
    "Nathaniel": "Nathaniel Yoon",
    "Natali Saraf": "Natali Sarraf",
    "Nat Pupattanakajor N": "Nat Puappatankajorn",
    "Nash Reibe": "Nash Riebe",
    "Mohammad Sarker": "Mohammed Sarker",
    "Misimi Sanni": "Mismi Sanni",
    "Michelle Li": "Michelle Liu",
    "Michael Schermerhorn": "Mike Schermerhorn",
    "Michael Hanson": "Michael Hansen",
    "Kevin Hammil": "Kevin Hammill",
    "Micah": "Micah Kawecki",
    "Mckinley Cherrier": "McKinley Cherrier",
    "Maxwell Taborrok": "Maxwell Tabarrok",
    "Max Wiener": "Maxwell Weiner",
    "Nico Llorente": "Nico Llorente Valin",
    "Travis Hunsburger": "Travis Hunsberger",
    "Ian Gate": "Ian Gates",
    "Mounisha Anumolo": "Mounisha Anumolu",
    "Karan Kuppa-Ape": "Karan Kappa-Apte",
    "Alex Hellinghausen": "Alexandra Hellinghausen",
    "Mac O’Hara": "Mac O'Hara",
    "Amarachi Alozie": "Amarchi Alozie",
    "Bev Soriano": "Bey Soriano",
    "Max Tabarrok": "Maxwell Tabarrok",
    "Max Sheremata Gu": "Maxwell Sheremeta",
    "Matthew Rubensten": "Matt Rubenstein",
    "Matthew Rubenstei N": "Matt Rubenstein",
    "Mathew Moreno": "Matthew Moreno",
    "Mario Aguire": "Mario Aguirre",
    "Mariana Icaza-Diaz": "Mariana Icaza Diaz",
    "Manuel Machorro Gomez Pezuela": "Manuel Machorro",
    "Madison Damien": "Madison Damian",
    "Madeleine Eichhorn": "Madeleine Eichorn",
    "Maddy Chesire": "Maddy Cheshire",
    "Maddie Nagel": "Maddie Nagle",
    "Mackenzi Tran": "Mackenzie Tran",
    "Mac O'Hare": "Mac O'Hara",
    "Mac Hayes": "Mac Hays",
    "Mabel Reiger": "Mabel Rieger",
    "Lê Quang Trịnh": "Quang Trinh",
    "Lydia Vlastro": "Lydia Vlasto",
    "Luke Joel Shankar": "Joel Shankar",
    "Lizzie Mccord": "Lizzie McCord",
    "Lizz Kim": "Liz Kim",
    "Lindsey Gradowsk I": "Lindsey Gradowski",
    "Lindsey Gradow": "Lindsey Gradowski",
    "Lily Levin": "Lilly Levin",
    "Liliana D’Aguiar": "Liliana D'Aguiar",
    "Lesley Munenyasha Machimbidza": "Lesley Machimbidza",
    "Leandro Guevara Neyra": "Leandro Guevara-Neyra",
    "Khylan Stubblefiel D": "Khylan Stubblefield",
    "Khylan Stubblefi Eld": "Khylan Stubblefield",
    "Ibtihal Gassem": "Ibithal Gassem",
    "Gabriel Frank-Mcpheter": "Gabriel Frank-McPheter",
    "Yaroslav Opansyuk": "Yaroslav Opanasyuk",
    "Zachary Fedyk": "Zach Fedyk",
    "Sam Arneson": "Sam Arnesen",
    "Kensington Speer": "Kensington Spear",
    "Kcale Teevan": "Cale Teevan",
    "Sunint Bundra": "Sunint Bindra",
    "Alice Merolli": "Alice Meroli",
    "Anshul Khakhar": "Anshul Khakar",
    "Anushri Swivedi": "Anushri Dwivedi",
    "Ariana Sharifi": "Ariane Sharifi",
    "Avrey Li": "Avery Li",
    "Caitlyn Jaeyeon Kim": "Caitlyn Kim",
    "Catie Macaulay": "Catie Mccauley",
    "Cayleigh Solderholm": "Cayleigh Soderholm",
    "Cecelia Szkutak": "CeCe Szkutak",
    "Cees Armstrong": "Coen Armstrong",
    "Christian Seskosan": "Christian Sekosan",
    "Claire Fennel": "Claire Fennell",
    "Colin Mac Hays": "Mac Hays",
    "Cody Magnus": "Cody McManus",
    "Curtis Lee": "Kurtis Lee",
    "Nicholas Milan": "Nicolas Millan",
    "Kayla Chen": "Kayla Chan",
    "Karan Kuppa-Apte": "Karan Kappa-Apte",
    "Jullia Chanda": "Jullian Chanda",
    "Julia W.": "Julia Wang",
    "Judy Jiang": "Judie Jiang",
    "Joshua Neudorf": "Josh Neudorf",
    "Joshua Kretchmer": "Josh Kretchmer",
    "Josh Tandiono": "Joshua Tandiono",
    "Josh Josheph": "Josh Joseph",
    "Joseph Bilotta": "Joe Bilotta",
    "Jorge Arturo Ramirez": "Jorge Ramirez",
    "Joe Billotta": "Joe Bilotta",
    "Jillian Elkins-Brumwell": "Jillian Elkins",
    "Jess Wang": "Jessica Wang",
    "Jenn Tran": "Jennifer Tran",
    "David Morales Lam": "David Morales",
    "David Ruvagaa": "David Ruvaga",
    "Dea Karemeti": "Dea Karameti",
    "Desmund Hui": "Desmond Hui",
    "Devesh Kodani": "Devesh Kodnani",
    "Dhafer Muhammed": "Dhafer Muhammad",
    "Dhruva Sumeshwar": "Dhruva Someshwar",
    "Dom Passafium E": "Dom Passafiume",
    "Dominic Digioia": "Dom Digioia",
    "Drew Meseck": "Drew Mesek",
    "Eden Rowe": "Eden Row",
    "Elijajh Nelson": "Elijah Nelson",
    "Elisa Gonzales": "Elisa Gonzalez",
    "Elizabeth Burkle": "Elizabeth Buerkle",
    "Emilio Stuart-Alb An": "Emilio Stuart-Alban",
    "Emily Zheng": "Emily Zhang",
    "Emma Jean Hermacinski": "EJ Hermacinski",
    "Erika": "Erika McCague",
    "Ethan James Mcminn": "Ethan James McMinn",
    "Ethan Mcminn": "Ethan James McMinn",
    "Gautam Ramasamy": "Gautum Ramasamy",
    "Erica Dinapoli": "Erica DiNapoli",
    "Elliot Mokski": "Elliott Mokski",
    "Ethan Rosenbaum": "Evan Rosenbaum",
    "Ethan Shurburg": "Ethan Shurberg",
    "Ezza Tariq": "Ezzah Tariq",
    "Fabi Ortez": "Fabbi Ortez",
    "Gabi Cunningha M": "Gabi Cunningham",
    "Genesis Lopez De Leon": "Genesis Lopez",
    "Gorbo Shilcusky": "Gabbi Shilcusky",
    "Gordon Mcneil": "Gordon McNeill",
    "Grace Flyyn": "Grace Flynn",
    "Grace Mctigue": "Grace McTigue",
    "Sophia Tyrell-Knott": "Sophia Tyrrell-Knott",
    "Alex Gellman-Beer": "Alex Gellman",
    "Greg Gentil": "Greg Gentile",
    "Griffin Badlamente": "Griffin Badalamente",
    "Grishma Baraugh": "Grishma Baruah",
    "Gwendolyn Havern": "Gwen Havern",
    "Habiba Mbugua": "Habib Moody",
    "Harrison Lavelle": "Harisson Lavelle",
    "Helen W": "Helen Wu",
    "Himnashu Padnani": "Himanshu Padnani",
    "I'Yanna Jones": "I'yanna Jones",
    "Ioannis Kryiakou": "Ioannis Kyriakou",
    "Christopher Vanderpool": "Chris Vanderpool",
    "Isa Irvine": "Isabel Irvine",
    "Jack Manimalco": "Jack Maniscalco",
    "Jake Fenster": "Jacob Fenster",
    "Jake Wasinger": "Jacob Wasinger",
    "James Eiferman": "James Eigerman",
    "Jane Metzinger": "Jane Mentzinger",
    "Janul Da Silva": "Janul de Silva",
    "Jarden Lenn": "Jared Lenn",
    "Jay Lewis": "James Lewis",
    "Jeffery Sheng": "Jeffrey Sheng",
    "Phil Maniscalco": "Phillip Maniscalo",
    "Jehan Chanmugan": "Jehan Chanmugam",
    "Karina Wugang": "Karina Wuwang",
    "Fletcher Calcagano": "Fletcher Calcagno",
    "Lem Yu": "Lemuel Yu",
    "Sándor Erik Lorange": "Sándor Lorange",
    "Sandor Lorange": "Sándor Lorange",
    "Reverand Sandor Lorange": "Sándor Lorange",
    "Eden Rowe": "Eden Row",
    "Alexander Bennett": "Alex Bennett",
    "Rafi Chowdurry": "Rafi Chowdhury",
    "Vedant Kerjariwal": "Vedant Kejariwal",
    "Elizabeth A Chen": "Elizabeth Chen",
    "Alessandro Perri": "Ale Perri",
    "Dan Rochon": "Daniel Rochon",
    "Joshua Cohen": "Josh Cohen",
    "Simran Verma-Singh": "Simran Singh",
    "Leemah B.": "Leemah Bisht",
    "Eva Marie-Quinones": "Eva-Marie Quinones",
    "Dixsheta Muralikrishnan": "Dixie Muralikrishnan",
    "Gabe Salgado": "Gabriel Salgado",
    "Benjamin Scherzer": "Ben Scherzer",
    "Danny Lee": "Denny Lee",
    "Sándor Erik Lorange":"Sándor Lorange",
    "Samuel Slack": "Sam Slack",
    "Caroline Sagristano": "Caroline Sagristino",
    "Bella Campbell": "Annabella Campbell",
    "Nathan Sears": "Natt Sears",
    "Nicolas Llorente Valin": "Nico Llorente Valin",
    "Hannah Platter": "Hannah Platter",
    "Eunisa Lu": "Eunisa Liu",
    "Eva Bruce ": "Eva Bruce",
    "Devank Agarwal": "Devansh Agarwal",
    "Eddie Sidoxrenko": "Eddie Sidorenko",
    "Kathryn Perrone": "Kate Perrone",
    "Zoe Savoy Rose": "Zoe Rose",
    "Anisah Colon": "Anisah Colón",
    "Katie Mae Ryan": "Katherine Ryan",
    "Kate Santarelli": "Katie Santarelli",
    "Nico Llorente-Valin": "Nico Llorente Valin",
    "Ícaro Teixeira": "Icaro Teixeira",
    "Jess Mcelroy": "Jessica Mcelroy",
    "Genevieve Savage": "Geneveive Savage",
    "Kiran Subramaniam": "Kiran Subramanian",
    "Sanket Bhaloti": "Sanket Bhalotia",
}

Secondarily, I used matching to examine the schools of past partners of unidentified debaters in order to determine the school they likely came from.

In [1583]:
recovered_debaters = [ 
    {"Debater": "Alex Chaparro",         "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Anthony Bragoli",       "School": "University of Virginia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Clara Yang",            "School": "Williams",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Isabela Eliassen",      "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jana Kassem",           "School": "University of Chicago",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Joey Deleone",          "School": "Smith",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Judy Liu",              "School": "Georgetown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Matt Seitz",            "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jack Anschultz",        "School": "University of Massachusetts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Isaiah Minter",         "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Imaan Chaudhry",        "School": "Smith",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ian Gates",             "School": "University of Virginia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Hikaru Hayakawa",       "School": "Tufts",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Henry Combs",           "School": "University of Chicago",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Germalysa Ferrer",      "School": "Princeton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Gabriel Mock",          "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ellie Park",            "School": "Boston University",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Eleni Antoniades",      "School": "Haverford",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Dylan Greenspan",       "School": "Binghamton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Daniel Coughlin",       "School": "Binghamton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Avina Sharma",      "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jonathan Morse",        "School": "University of Massachusetts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Kevin Hammill",         "School": "Fordham",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Samridhi Parasrampuria","School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Tania Acsinte",         "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Mila Maglov",           "School": "Brandeis",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Katrina Deng",          "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Julia Sicard",          "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jude Hoag",             "School": "Boston University",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Joyce Choi",            "School": "Georgetown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jane Kassemk",          "School": "University of Chicago",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Kiran Das-Goel",        "School": "Smith",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Lily Kwak",             "School": "Columbia",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Malak Hajiyeva",        "School": "University of Massachusetts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Matt Pekor",            "School": "University of Pittsburgh",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Maura Baker",           "School": "Haverford",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Michael Hoffman",       "School": "George Washington",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Michael Okpoti",        "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Pranav Sundar",         "School": "Haverford",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Reece Danzis",          "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ronnie Hokett",         "School": "Binghamton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Sam Brown",             "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sefat Uddin Samee",     "School": "Tufts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jay Mathur",            "School": "Maryland",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jay Philbrick",         "School": "Brown",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Jeffrey Stein",         "School": "Columbia",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jennie Fan",            "School": "Penn",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jennifer Lin",          "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Joe Clark",             "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jonathan Iacovacci",    "School": "Maryland",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Yun Zhang",             "School": "Bates",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Vedant Kejariwal",      "School": "Boston University",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Tyler Swartz",          "School": "Rutgers",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Taijah Chavis",         "School": "Boston University",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jack Tajmajer",         "School": "Brown",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Jackson Kramp",         "School": "Lehigh",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Jacob Weinberg",        "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jack Shapiro",          "School": "Maryland",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jacoby Sypher",         "School": "George Washington",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Jacqueline Wang",       "School": "Brandeis",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Jade Ye",               "School": "Bentley University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jai Kovvuri",           "School": "Pace",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Jaime Colon",           "School": "Northeastern",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Jake Bohman",           "School": "Swarthmore",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jake Lesser",           "School": "University of Pittsburgh",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Jamie Davis",           "School": "George Washington",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jana Jandal Alrifai",   "School": "Tufts",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Jaqueline Rizzi",       "School": "Boston University",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jaren Friesen",         "School": "Boston University",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jason Alder",           "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jay Kim",          "School": "William and Mary",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Ian Bass",             "School": "Rutgers",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Igal Sultanov",        "School": "CUNY",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Ilana Gellman",        "School": "University of Chicago",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ilina Logani",         "School": "Columbia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Iman Obargi",          "School": "NYU",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Irene Kim",            "School": "Yale",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Iris Kim",             "School": "Haverford",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Isaac Pedersen",       "School": "Northeastern",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Isabella Franklin",    "School": "NYU",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ishan Patel",          "School": "Georgetown",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Israel Pierre",        "School": "University of Chicago",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Issa Sadamoto",        "School": "Stanford",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Iveena Mukherjee",     "School": "Penn",   "Initial Year": 2026, "Last Year Seen": 2026},
    {"Debater": "Jack Hunt",            "School": "Villanova",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jack Shapiro",          "School": "Maryland",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Hyunsoo Lee",          "School": "UMBC",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Hugo Hinze",           "School": "Harvard",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Hong Meng Yam",        "School": "Stanford",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Holly Dickinson",      "School": "Smith",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Hiyanshi Patel",       "School": "Maryland",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Hermione Cordeiro Larkin","School": "Johns Hopkins",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Henry Ren",            "School": "Georgetown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Henry Lei",            "School": "Swarthmore",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Hemanth Asirvatham",   "School": "Harvard",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Helen Phi",            "School": "Brown",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Hayley Yeung",         "School": "University of Chicago",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Hayk Kibarian",        "School": "American",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Harris Agha",          "School": "Williams",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Harisson Lavelle",     "School": "University of Virginia",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Hannah To",            "School": "Princeton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Hannah Messaye",       "School": "Amherst",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Haneul Shin",          "School": "Boston University",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Hamza Kalim",          "School": "Bates",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Billy Donoso",         "School": "Tufts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Blake Lukens",         "School": "Bentley University",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Caitlin Han",          "School": "Tufts",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Chelsea Lugat",        "School": "Binghamton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Christopher Chiu",     "School": "Villanova",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "David Choi",           "School": "Penn",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "David Timmerman",      "School": "Binghamton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Destiny Eversole",     "School": "Wellesley",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Diyor Kamolov",        "School": "Columbia",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Drew Bennison",        "School": "University of Virginia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ethan Lin",            "School": "Amherst",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Guangxinyang Deng",    "School": "University of Chicago",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Gwen Havern",          "School": "Haverford",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Gwen Stearns",         "School": "Johns Hopkins",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Habikah Baldeh",       "School": "Johns Hopkins",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Hagan Werner",         "School": "George Washington",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Hai Tran",             "School": "University of Massachusetts",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Hailey Demars",        "School": "Stanford",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Hailey Lorence",          "School": "American",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Haitong Du",          "School": "George Washington",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Aaron Kopew",          "School": "Rutgers",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Abhishek Girish",      "School": "NYU",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Adam Usmanov",         "School": "University of Pittsburgh",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Adeolu Ajayi",         "School": "Columbia",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Aidan Kuk",            "School": "Johns Hopkins",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Alex Gellman",         "School": "William and Mary",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ambika Kandasemy",     "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Anna Izyumova",        "School": "Princeton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ayushi Singh",         "School": "University of Massachusetts",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ben Luo",              "School": "University of Chicago",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ben Xu",               "School": "Tufts",   "Initial Year": 2019, "Last Year Seen": 2019},   
    {"Debater": "Nicole Berglund",      "School": "University of Massachusetts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Nicolas Howayeck",     "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Nico Cavalluzzi",      "School": "Boston University",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Nathan Olski",         "School": "University of Virginia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Maya Hoffman",         "School": "Georgetown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Martin Lapczyk",       "School": "NYU",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Margaret Macgillivray","School": "Williams",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Makal Matthews",       "School": "Prince George's Community College",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Maia Harrison",        "School": "Princeton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Lindsay Lake",         "School": "Brown",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Lena Sernoff",         "School": "NYU",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Lani Craig",           "School": "William and Mary",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Kylie Kim",            "School": "Boston University",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Keevon Thomas",        "School": "Temple",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kate Perrone",         "School": "University of Virginia",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "John Lynch",           "School": "William and Mary",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Stephanie Lee",        "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Trevor Cope",          "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Yi Huang",             "School": "Maryland",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sophia Cruz",          "School": "Grinnell",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Shayan Reza",          "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Serkute Abebe",        "School": "Columbia",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Seha Karabacak",       "School": "Tufts",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Santosh Gontla",       "School": "Davidson",   "Initial Year": 2026, "Last Year Seen": 2026},
    {"Debater": "Sam Wing",             "School": "Binghamton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Sam Hano",             "School": "University of Massachusetts",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sal Conte",            "School": "Bentley University",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ruhaan Chopra",        "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ronald Taylor",        "School": "Grinnell",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Robert Summers Berger","School": "Carnegie Mellon",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Rahi Patel",           "School": "University of Massachusetts",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Raegan Arroyo",         "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Onder Kilinc",         "School": "University of Chicago",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Gaston Aime",          "School": "Tufts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Gaurav Bagur",         "School": "Brandeis",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Gavin Roulett",        "School": "William and Mary",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Gena Rising",          "School": "Binghamton University",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Genesis Lopez",        "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Geneveive Savage",     "School": "Northeastern",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "George Harrison",      "School": "University of Chicago",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Georgedaniel Dixon",   "School": "Amherst",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Gianna Bruno",         "School": "Brandeis",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Gianna Naulivou",      "School": "University of Massachusetts",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Gladwin An",           "School": "Rutgers",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Govind Menon",         "School": "Carnegie Mellon",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Grace Shamer",         "School": "William and Mary",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Grace Wang",           "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Grace Wu",             "School": "Boston University",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Graham Sagel",         "School": "Tufts",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Dylan Wallerstein",    "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Elizabeth Qiao",       "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Florence Nelson",      "School": "Prince George's Community College",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Francisca Wijaya",     "School": "Wesleyan",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Frank Chiu",           "School": "Brown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Frank Parizek",        "School": "William and Mary",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Gabe Cronin-Golomb",   "School": "Smith",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Gabriel Uceda-Sosa",   "School": "Columbia",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Gabriel Young",        "School": "George Washington",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Gabriela Roznawska",   "School": "Grinnell",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Gaby Ivanova",           "School": "University of Chicago",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Garvin Kim",           "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Emily Dale",           "School": "Princeton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Eric Parrilla",        "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ethan Quinn",          "School": "Temple",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Eva Bingham",          "School": "George Washington",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Evan Forrest",         "School": "University of Massachusetts",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Evan Michaels",        "School": "American",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Evelyn Chen",          "School": "Northeastern",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Evyn Appel",           "School": "University of North Carolina",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ewa Tryniszewski",     "School": "Georgetown",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Ezzat Abouleish",      "School": "Yale",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Faith Ale",            "School": "Rutgers",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Fiona Yang",           "School": "Maryland",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Flavia Maria Galeazzi","School": "Brown",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "James Coppersmith",    "School": "Columbia",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jason Ramdeo",         "School": "William and Mary",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Jemima Williams",      "School": "Princeton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Emmanuel Yamba",       "School": "George Washington",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Hector Hernandez",     "School": "Tufts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jailam Hutton",        "School": "Temple",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Jake Macnelly",        "School": "Boston College",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Dylan Safai",          "School": "Williams",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Elise Greene",         "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Elizabeth Rengifo",    "School": "Fordham",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Elizabeth Rice",       "School": "Bentley University",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Elizaveta Bakhtina",   "School": "William and Mary",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Erin Hunter",          "School": "University of Massachusetts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Ethan Brown",          "School": "Hamilton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Erin Howard",          "School": "Wellesley",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ethan Carter",         "School": "NYU",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ethan Dubinsky",       "School": "Fordham",   "Initial Year": 2022, "Last Year Seen": 2025},
    {"Debater": "Ethan Knox",           "School": "Penn",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Eric Trueswell",       "School": "University of Massachusetts",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Eunisa Liu",           "School": "Maryland",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Gabriel Salgado",      "School": "University of Pittsburgh",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Hailey Baker",          "School": "Fordham",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Jaelyn Perez",          "School": "University of Massachusetts",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Jake Grande",          "School": "George Washington",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "James Elmore",          "School": "William and Mary",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Jake Sledge",          "School": "Princeton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Ella Bullock-Papa",    "School": "Stanford",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ella Scott",           "School": "Washington University in St. Louis",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ellie Berenson",       "School": "William and Mary",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Elliot Jones",         "School": "Wellesley",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Emanuel Yamba",        "School": "George Washington",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Emily Flood",          "School": "Harvard",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Emily Herstine",       "School": "Temple",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Emily Kuchuk",         "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Emma Mcdonough",       "School": "Northeastern",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Emma Mcfall",          "School": "Brown",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Emma Smith",           "School": "Georgetown",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Emmanuel Abay",        "School": "Davidson",   "Initial Year": 2026, "Last Year Seen": 2026},
    {"Debater": "Erica Fisher",         "School": "University of Pittsburgh",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Erica Otte",           "School": "Maryland",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Erika Vasquez-Rivas",  "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Erin Converse",        "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Erin Howard",          "School": "Wellesley",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Eden Row",             "School": "Columbia",   "Initial Year": 2020, "Last Year Seen": 2021},
    {"Debater": "Edona Cosovic",        "School": "Harvard",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Edward Frazer",        "School": "Yale",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Edward Orji",          "School": "Northeastern",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Edward Yang",          "School": "NYU",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Efe Alpay",            "School": "Brown",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Eileen Qiu",           "School": "Brandeis",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ej Beck",              "School": "University of Chicago",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Elaina Craig",         "School": "Fordham",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Elaine Wang",          "School": "Brown",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Elena Prisament",      "School": "MIT",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Elizabeth Chen",       "School": "CUNY",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Elizabeth Janes",      "School": "Lehigh",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Elizabeth Kean",       "School": "Georgetown",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Elizabeth Kim",        "School": "Boston University",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Elizabeth Morison",    "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alan Pham",            "School": "Smith",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Carley Medeiros",      "School": "Johns Hopkins",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Derek Zhen",           "School": "Washington University in St. Louis",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Dev Patel",            "School": "Maryland",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Devin Mullen",         "School": "Rutgers",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Devin Pracar",         "School": "Carnegie Mellon",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Dheeraj Pasikanti",    "School": "George Washington",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Dhruv Kohli",          "School": "University of Chicago",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Diana Gothong",        "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Diego Estrada Adame",  "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Dixie Muralikrishnan", "School": "Swarthmore",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Diyor Kamalov",        "School": "Columbia",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Do Nguyen Tung",       "School": "Princeton",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Dovran Babayev",       "School": "Fordham",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Drew Morehead",        "School": "Brown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Drew Thomas",          "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Dung Nguyen",          "School": "Johns Hopkins",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "E'Niah Preston",       "School": "Rutgers",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Ebenezer Appiah",      "School": "Brown",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Dean Walters",         "School": "University of Pittsburgh",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "David Dai",            "School": "Harvard",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "David Borawski",       "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Daria Mitri",          "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Dany Matar",           "School": "Washington University in St. Louis",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Danny Welden",         "School": "Pace",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Danilo Garcia",        "School": "CUNY",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Clarissa Dias",        "School": "Amherst",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Brian Forgue",         "School": "University of Massachusetts",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Avi Konduri",          "School": "Columbia",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alexandra Hellinghausen","School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Achuthan Panikath",    "School": "University of Massachusetts",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Abielle Ahn",          "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jennifer Tran",        "School": "Rutgers",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jenny Chen",           "School": "Johns Hopkins",   "Initial Year": 2021, "Last Year Seen": 2022},
    {"Debater": "Jeremiah Harrington",  "School": "Bates",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jeremy Evans",         "School": "Brandeis",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Jessica Johnson",      "School": "Smith",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Jessica Mcelroy",      "School": "Rutgers",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jessie Zhou",          "School": "Smith",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jewel Thomas",         "School": "University of Virginia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jiahao Guo",           "School": "Georgetown",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Joey Arniel",          "School": "Bates",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jielu Yu",             "School": "University of Chicago",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jillian Elkins",       "School": "Brandeis",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Jimmy Galvin",         "School": "University of Virginia",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jimmy Goranov",        "School": "University of Virginia",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Jimmy Pham",           "School": "Swarthmore",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Jin Huang",            "School": "University of Chicago",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Jingchen Peng",        "School": "NYU",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jinge Cao",            "School": "Boston University",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Jocelyn Gao",          "School": "Rutgers",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Joe Anderson",         "School": "Penn",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Joe Maalouf",          "School": "Hamilton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Joe Patti",            "School": "Columbia",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Joey Cano",            "School": "Amherst",   "Initial Year": 2026, "Last Year Seen": 2026},
    {"Debater": "Joey Juul God",        "School": "Northeastern",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Johann Lindberg",      "School": "William and Mary",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "John Ablonczy",        "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "John Godwin",          "School": "University of Virginia",   "Initial Year": 2021, "Last Year Seen": 2022},
    {"Debater": "John Polito",          "School": "Georgetown",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Jonah Wunder",         "School": "American",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Jonathan Pinelli",     "School": "Tufts",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Jordan Medved",        "School": "Brandeis",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Joseph Dey",           "School": "University of Chicago",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Josephine Kuo",        "School": "Tufts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Josh Book",            "School": "Columbia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Josh Cohen",           "School": "Tufts",   "Initial Year": 2021, "Last Year Seen": 2024},
    {"Debater": "Josh Joseph",          "School": "Brandeis",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Joshua Ehizibolo",     "School": "Maryland",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Julia Brous",          "School": "Hamilton",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Julia Dietrich",       "School": "University of Pittsburgh",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Julia Karabolli",      "School": "UMBC",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Julio Cordero",        "School": "CUNY",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Juno Tantipipatpong",  "School": "Brown",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Justin Palus",         "School": "Rutgers",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Justin Shillingford",  "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2025},
    {"Debater": "Justin Woo",           "School": "Brown",   "Initial Year": 2020, "Last Year Seen": 2021},
    {"Debater": "Kanika Mehra",         "School": "Maryland",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kanisha Harrell",      "School": "Johns Hopkins",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Karan Makkar",         "School": "Haverford",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Karen Kao",            "School": "Smith",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Karim Zohdy",          "School": "Brown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Kate Davis",           "School": "University of Chicago",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Kate Selig",           "School": "Stanford",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Kate Stover",          "School": "Fordham",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Katelyn Rickert",      "School": "Georgetown",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Katherine Ryan",       "School": "Carnegie Mellon",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Kathy Piperno",        "School": "Rutgers",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Katie Santarelli",     "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kaydra Hopkins",       "School": "Northeastern",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Kayla Costa",          "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kayla Morrison",       "School": "Brown",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Mansi Bahl",           "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Margo Mandell",        "School": "Georgetown",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Nisha Athrey",         "School": "Georgetown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Noah Lennon",          "School": "Binghamton University",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Paul Marcelli",        "School": "Johns Hopkins",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Raaid Khan",           "School": "Lehigh",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sahil Gaba",           "School": "Georgetown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Shayan Raza",          "School": "University of Massachusetts",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Sylvie Mahoro",        "School": "Bates",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Daniel Welden",        "School": "Pace",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Daniel Rochon",        "School": "George Washington",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Daniel Leizerman",     "School": "Bates",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Daniel Haskell",       "School": "Bowdoin College",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Dan Leizerman",        "School": "Bates",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Damian Vladmiroff",    "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Curtis Mcmackin",      "School": "Maryland",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Curtis Everett",       "School": "Harvard",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Cristiana Ramos",      "School": "Boston University",   "Initial Year": 2024, "Last Year Seen": 2024},  
    {"Debater": "Cormac Kimberly",      "School": "Yale",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Conor Bifulco",        "School": "NYU",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Colten Edelman",       "School": "Brown",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Claire Paul",          "School": "Boston University",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Chuck Foreman",        "School": "Binghamton University",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Christopher Owen",     "School": "William and Mary",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Christopher Lynch",    "School": "Dartmouth",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Christina Wang",       "School": "Bates",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Christian Sekosan",    "School": "Binghamton University",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Christian Gentolia",   "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2023},  
    {"Debater": "Carlos Freyre",        "School": "William and Mary",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Chapin Fish",          "School": "Fordham",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Charitha Vennapusa",   "School": "University of Chicago",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Charles Tufenkji",     "School": "Amherst",   "Initial Year": 2026, "Last Year Seen": 2026},
    {"Debater": "Chase Bezonsky",       "School": "University of Massachusetts",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Chelsea Long",         "School": "Brown",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Cheryl Minde",         "School": "Smith",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Chris Francis Anto",   "School": "Johns Hopkins",   "Initial Year": 2022, "Last Year Seen": 2022},    
    {"Debater": "Carlo Puca",           "School": "University of Chicago",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Caleb Shook",          "School": "University of Pittsburgh",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Brendan Heaney",       "School": "Binghamton University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Blake Mcneely",        "School": "American",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Bianca Rozario",       "School": "Boston University",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ben Scharr-Weiner",    "School": "Tufts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ben Robertson",        "School": "Brandeis",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ben Fligelman",        "School": "Haverford",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ayanna Williams",      "School": "William and Mary",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ava Peifer",           "School": "William and Mary",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Askar Mirza",          "School": "Rutgers",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ashna Guha",           "School": "University of Massachusetts",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Asher Moss",           "School": "Villanova",   "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Anvi Shettigar",       "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Anna Harshman",        "School": "William and Mary",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Angie Guo",            "School": "Brandeis",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Amarchi Alozie",       "School": "Bates",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Allison Buehler",      "School": "American",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Alex Hotzpaffel",      "School": "William and Mary",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Akkshansh Bagga",      "School": "Amherst",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Aiden Dowd",           "School": "University of Virginia",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Advait Ganapathy",     "School": "Harvard",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Adrian Kim",           "School": "Rutgers",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Adrian Brown",         "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Karan Kappa-Apte",     "School": "Bates",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Karen Li",             "School": "Wellesley",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Kaylah Costa",         "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kayleigh Hernandez",   "School": "Washington University in St. Louis",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Keith Do",             "School": "Wesleyan",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Kelvin Powell",        "School": "Columbia",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Kenneth Chen",         "School": "Harvard",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Kiran Das Goel",       "School": "Smith",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Malika Buribayeva",    "School": "Lehigh",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Marcelo Gonzales",     "School": "George Washington",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Max Markel",           "School": "William and Mary",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Mia Flaherty",         "School": "American",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Mikael Zarett",        "School": "Stanford",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Miles Katz",           "School": "NYU",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Naomi O'Meara",        "School": "Rutgers",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Noura Ag",             "School": "Brandeis",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Owen Meroski",         "School": "University of Massachusetts",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Philip Surendran",     "School": "Dartmouth",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Rishi Hazra",          "School": "Harvard",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ryan Zheng",           "School": "Boston College",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Salma Sheikh",         "School": "Rutgers",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Sarah Loewecke",       "School": "Fordham",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Simar Soni",           "School": "Penn",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Syed Azan Ali",        "School": "Georgetown",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Tushar Dalmia",        "School": "University of Chicago",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Uriah Colegrove",      "School": "Columbia",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Victoria Hagen",       "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Vik Georgieva",        "School": "Wesleyan",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "William Pirone",       "School": "University of Chicago",   "Initial Year": 2023, "Last Year Seen": 2025},
    {"Debater": "Hana Hussain",         "School": "Lehigh",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Icaro Teixeira",       "School": "Williams",   "Initial Year": 2025, "Last Year Seen": 2026},
    {"Debater": "Isabella Rocco",       "School": "Johns Hopkins",   "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Jack Keating",         "School": "William and Mary",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Jada Badillo",         "School": "Smith",   "Initial Year": 2019, "Last Year Seen": 2022},
    {"Debater": "Kenyatta Heavlow",     "School": "University of Massachusetts",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Kevin Cho",            "School": "Rutgers",   "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Kevin Khadavi",        "School": "University of Chicago",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Kevin Xue",            "School": "Penn",   "Initial Year": 2024, "Last Year Seen": 2026},
    {"Debater": "Khadeeja Qureshi",     "School": "Bates",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Khanh Doan",           "School": "American",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Khuslen Tulga",        "School": "Hamilton",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kingston Lew",         "School": "MIT",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Krishin Wadhwani",     "School": "Carnegie Mellon",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Krishma Gewali",       "School": "Columbia",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Kristina Megerdichian","School": "Tufts",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Kuangye Wang",         "School": "Columbia",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kyle Coopersmith",     "School": "University of Pittsburgh",   "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Kyle Quinlan",         "School": "Yale",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Kylie Feliciano",      "School": "Boston University",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Kyra Haddad",          "School": "Brown",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Lamisa Khan",          "School": "NYU",   "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Latifa Fasla",         "School": "Brandeis",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Laura Howard",         "School": "University of Virginia",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Laura Zhang",          "School": "Princeton",   "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Lauren Fanter",        "School": "Temple",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Lauren Kim",           "School": "Wellesley",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Lavina Ngo",           "School": "Smith",   "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Leander Quiroz",       "School": "Wellesley",   "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Leemah Bisht",         "School": "Maryland",   "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Leia Park",            "School": "Carnegie Mellon",   "Initial Year": 2022, "Last Year Seen": 2022},  
    {"Debater": "Leigh Murphy",         "School": "Williams",   "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Juan Diego Cisneros",  "School": "Lehigh",   "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Pragna Yalamanchili",  "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Bao Nghi Ho",          "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Aaliyah Bullen",       "School": "Swarthmore", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Aaradhya Diwan",       "School": "Harvard", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Kiran Subramanian",    "School": "Rutgers", "Initial Year": 2021, "Last Year Seen": 2024},
    {"Debater": "Frank Rodriguez",      "School": "Northeastern", "Initial Year": 2019, "Last Year Seen": 2023},
    {"Debater": "Oden",                 "School": "NYU", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Leon Gold",            "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2024},
    {"Debater": "Abby Sweeney",         "School": "George Washington", "Initial Year": 2020, "Last Year Seen": 2022},
    {"Debater": "Chris Vanderpool",     "School": "Brown", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Zora Kuehne",          "School": "Haverford", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Yese Erazo-Tequianes", "School": "Pace", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Aatikah Awan",         "School": "NYU", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Abby Garber",          "School": "Wellesley", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Abby Morin",           "School": "American", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Abhinav Agarwal",      "School": "Princeton", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Abhinav Aitha",        "School": "Fordham", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Abigail Romero",       "School": "Harvard", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Abrar Ahmed",          "School": "Columbia", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Adam Gould",           "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Adam Rusakow",         "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Adi Chattopadhyay",    "School": "Swarthmore", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Adrian Seferlis",      "School": "Rutgers", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Adrien Amouroux",      "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Advesh Jalan",         "School": "Yale", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ahan Raina",           "School": "University of Chicago", "Initial Year": 2021, "Last Year Seen": 2022},
    {"Debater": "Aidan Liu",            "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Aishwarya Rajapur",    "School": "Amherst", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Aj Matos",             "School": "Bates", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Akber Latif",          "School": "George Washington", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Alaska Irr",           "School": "Brandeis", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Albert Hao",           "School": "Columbia", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Aleeza Syed",          "School": "Northeastern", "Initial Year": 2023, "Last Year Seen": 2025},
    {"Debater": "Alekhya Bhat",         "School": "Wellesley", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Juan Diego Cisneros",  "School": "Lehigh", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Pragna Yalamanchili",  "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Bao Nghi Ho",          "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Alessandra Lorenzo",   "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alex Bennett",         "School": "Brandeis", "Initial Year": 2020, "Last Year Seen": 2022},
    {"Debater": "Alex Rizzo",           "School": "American", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Alex Shieh",           "School": "Brown", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alexa Sanchez",        "School": "Smith", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Alexander Mclaren",    "School": "Columbia", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alexander West",       "School": "Temple", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alexander Yankovsky",  "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Alexis Walker",        "School": "University of Virginia", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Alia Bonanno",         "School": "Columbia", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alia Derriey",         "School": "Binghamton University", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Alia Kafil",           "School": "Columbia", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alice Yang",           "School": "Drexel", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Alina Antropova",      "School": "University of Massachusetts", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Alina Palacios",       "School": "Swarthmore", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Allie Rosenstein",     "School": "Harvard", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Almila Muslu",         "School": "Drexel", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Amadou Thiam",         "School": "Prince George's Community College", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Amanda Blatz",         "School": "Haverford", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Amarynth Ruch",        "School": "Temple", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Amber Gao",            "School": "NYU", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ambika Kandasamy",     "School": "Johns Hopkins", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ameena Ahmed",         "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Ameia Booker",         "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Amelia Le",            "School": "Lehigh", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Anantha Kashibhatla",  "School": "Rutgers", "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Anastasia Malenko",    "School": "Stanford", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Andrea Zhou",          "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Andrew Cole",          "School": "University of Pittsburg", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Andrew Conkey",        "School": "University of Chicago", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Andrew Hoffman",       "School": "William and Mary", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Andrew Vandenbussche", "School": "Penn", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Angel Paz",            "School": "NYU", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Anirudh Narsipur",     "School": "Brown", "Initial Year": 2021, "Last Year Seen": 2022},
    {"Debater": "Anisah Colón",         "School": "Villanova", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Anish Kanthameni",     "School": "NYU", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Anjali Agarwal",       "School": "Haverford", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Anna Ast",             "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Anna Krans",           "School": "Yale", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Anna Nowalk",          "School": "Fordham", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Alison Linares",       "School": "Yale", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Annabella Campbell",   "School": "Fordham", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Annie Chi",            "School": "Princeton", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Annie Dong",           "School": "Penn", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Annie Zhi",            "School": "University of Chicago", "Initial Year": 2020, "Last Year Seen": 2022},
    {"Debater": "Anouk Yeh",            "School": "Yale", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Anshika Agrawal",      "School": "Johns Hopkins", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Anthony Boss",         "School": "Brown", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Anuka Upadhye",        "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Anya Rohatgi",         "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Aogay Alozai Wardak",  "School": "Tufts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Apollo Grimes",        "School": "Brandeis", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Ari Gershengorn",      "School": "NYU", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Ariane Sharifi",       "School": "Maryland", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Anya Rohtagi",         "School": "Wellesley", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Arjun Suryawanshi",    "School": "Penn", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Armaan Sheth",         "School": "University of Massachusetts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Aron Ravin",           "School": "Yale", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Arthur Yolles",        "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Arushi Kaushik",       "School": "NYU", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Arushi Sahay",         "School": "NYU", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Arya Nalluri",         "School": "American", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Asher Mendelson",      "School": "NYU", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Ashkay Srivasan",      "School": "Columbia", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Ashley Huang",         "School": "Tufts", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Athena Atsides",       "School": "George Washington", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Audri Bhomick",        "School": "Brandeis", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Austin Chapman",       "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Austin Chen",          "School": "Brandeis", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Austin Riegel",        "School": "Rutgers", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Austin Zheng",         "School": "Bowdoin College", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Autri Basu",           "School": "Amherst", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ava Desantis",         "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Ava Nagy",             "School": "Boston University", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Ava Rahman",           "School": "Brown", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Avery Lenihan",        "School": "Yale", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Avery Li",             "School": "William and Mary", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Awsam Bouabid",        "School": "Northeastern", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ayaan De Silva",       "School": "University of Chicago", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Ayva Kacir",           "School": "Tufts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Bairavi Sundaram",     "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Barna Marczali",       "School": "Johns Hopkins", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Becca Carer",          "School": "Johns Hopkins", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ben Bradley",          "School": "Brown", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ben Fica",             "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Ben Fitzgerald",       "School": "Haverford", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ben Hinish",           "School": "Drexel", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ben Skarbek",          "School": "Temple", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Ben Wilson",           "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ben Wong",             "School": "Binghamton University", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Benjamin Xu",          "School": "Williams", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Berk Turkkani",        "School": "NYU", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Betty Qian",           "School": "Wellesley", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Bey Soriano",          "School": "Johns Hopkins", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Bhavya Surapaneni",    "School": "Penn", "Initial Year": 2023, "Last Year Seen": 2024},    
    {"Debater": "Bianca Ferreira",      "School": "Temple", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Bilal Tariq",          "School": "Amherst", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Bill Chen",            "School": "Harvard", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Blair Peng",           "School": "University of Chicago", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Bradley Evans",        "School": "University of Pittsburgh", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Brandon Bosaz",        "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Breanna Crossman",     "School": "George Washington", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Brendan Brestage",     "School": "Northeastern", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Brennan Mcdermott",    "School": "American", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Brian O'Neill",        "School": "Tufts", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Brittany Bin",         "School": "MIT", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Bryan Mcdonough",      "School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Bryan Thomas",         "School": "Northeastern", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Bryce Trent",          "School": "Williams", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Cayden Monteiro",      "School": "George Washington", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Carley Calfee",        "School": "American", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Brynn Kroke",          "School": "Brown", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Chansol Park",         "School": "NYU", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Daisy Bateman",        "School": "American", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Fabiha Era",           "School": "Binghamton University", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "George Anderson",      "School": "William and Mary", "Initial Year": 2019, "Last Year Seen": 2021},
    {"Debater": "Hassan Looky",         "School": "Harvard", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Zachary Braunstein",   "School": "Maryland", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Tyler Hugo",           "School": "Temple", "Initial Year": 2022, "Last Year Seen": 2024},
    {"Debater": "Tess Yu",              "School": "Johns Hopkins", "Initial Year": 2021, "Last Year Seen": 2022},
    {"Debater": "Akhil Mallajosyula",   "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Callum Mcfarlane",     "School": "University of Chicago", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Cameron Sagheb",       "School": "Georgetown", "Initial Year": 2021, "Last Year Seen": 2022},
    {"Debater": "Camille Jones",        "School": "Princeton", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Chloe Yu",             "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Connor Beaney",        "School": "Brandeis", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Ellie Papraniku",      "School": "CUNY", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Evan Tao",             "School": "Brown", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Zoe Rose",             "School": "William and Mary", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Zoe Raptis",           "School": "Tufts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Yosua Siagian",        "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Yasmine Dweir",        "School": "NYU", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Varun Singh",          "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Wayland Bardwell",     "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2024},
    {"Debater": "Trevor Kickliter",     "School": "University of Pittsburgh", "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Davin Bhatti",         "School": "University of Chicago", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Clarity Houts",        "School": "Amherst", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Cleo Elrashidy",       "School": "Brown", "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Catherine Horner",     "School": "Dartmouth", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Carina Olivar",        "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Carme Sanz-Muñoz",     "School": "Wellesley", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Caroline Hennigan",    "School": "Harvard", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Caroline Loveday",     "School": "William and Mary", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Caroline Sagristino",  "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Claire Fennell",       "School": "NYU", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sriya Uttharkar",      "School": "Rutgers", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Sophia Mason",         "School": "Pace", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Smiti Modhurima",      "School": "Columbia", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Zoey Morris",          "School": "Stanford", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Zakiriya Gladney",     "School": "Harvard", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Yvette Shu",           "School": "Boston University", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Yuri Izumikawa",       "School": "Johns Hopkins", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Yuqian Li",            "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Yuntian Gan",          "School": "Brandeis", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Youssef Bousada",      "School": "NYU", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Yonah Gross",          "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Yoav Rafalin",         "School": "Carnegie Mellon", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Yinjun Chen",          "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Yilang Fan",           "School": "NYU", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Yijiao Guo",           "School": "Yale", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Yifei Sun",            "School": "Bentley University", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Yazeed Abayazid",      "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Yaxuan Li",            "School": "Carnegie Mellon", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Yasmin Roach",         "School": "Yale", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Yaroslav Opanasyuk",   "School": "CUNY", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Yannik Omictin",       "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Yannick Daoud",        "School": "NYU", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Yanni Trimikliniotis", "School": "NYU", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Yan Ning",             "School": "Brown", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Xinyue Gu",            "School": "Johns Hopkins", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Wyatt Smith",          "School": "Williams", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Winnie Jiang",         "School": "Yale", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Willow West",          "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Willie Gomez",         "School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "William Mcadams",      "School": "Washington University in St. Louis", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Will Florentino",      "School": "Georgetown", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Wasi Ahmed",           "School": "CUNY", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Wasan Rafat",          "School": "Harvard", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Walker Evans",         "School": "American", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Vinya Lingamneni",     "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Vincent Shen",         "School": "Rutgers", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Vik Li",               "School": "Northeastern", "Initial Year": 2019, "Last Year Seen": 2019},    
    {"Debater": "Vaughn Battista",      "School": "Rutgers", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Vara Qi Gunananthan",  "School": "Johns Hopkins", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Udi Akolkar",          "School": "Carnegie Mellon", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Tyler Klinedinst",     "School": "American", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Tuong Do",             "School": "Haverford", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Triet Le",             "School": "Fordham", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Trey Garcia-Schartz",  "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Trevor Haefner",       "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ting Hsu",             "School": "Boston University", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Tina Tang",            "School": "Georgetown", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Tim Brennan",          "School": "Binghamton University", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Thulasi Varatharajan", "School": "University of Pittsburgh", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Thomas Lin",           "School": "University of Chicago", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Theo Lockrow",         "School": "Wesleyan", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Thabang Matona",       "School": "Brandeis", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Tendai Coady",         "School": "Williams", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Tanaz Bari",           "School": "Binghamton University", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Tamaki Sugihara",      "School": "University of Massachusetts", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Talia Katz",           "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Tabitha Chua",         "School": "Boston University", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Renny Jiang",          "School": "Brown", "Initial Year": 2021, "Last Year Seen": 2023},
    {"Debater": "Riker Wachtler",       "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Peter Capote",         "School": "Rutgers", "Initial Year": 2024, "Last Year Seen": 2024},	
    {"Debater": "Omar Elalaoui",        "School": "Rutgers", "Initial Year": 2023, "Last Year Seen": 2024},	
    {"Debater": "Peter Heller",         "School": "William and Mary", "Initial Year": 2019, "Last Year Seen": 2021},	
    {"Debater": "Penelope Toll",        "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2022},	
    {"Debater": "Nyankpani Kesson Abdul-Quddud",         "School": "Boston University", "Initial Year": 2023, "Last Year Seen": 2023},	
    {"Debater": "Natalia Caid",         "School": "Temple", "Initial Year": 2024, "Last Year Seen": 2026},
    {"Debater": "Nafiz Zaman",          "School": "Johns Hopkins", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Mary Clarke",          "School": "Brown", "Initial Year": 2022, "Last Year Seen": 2025},
    {"Debater": "Mateo Mcnamara",       "School": "George Washington", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Mandy Feuerman",       "School": "Brandeis", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Madigan Webb",         "School": "William and Mary", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Sophie Mason",         "School": "Pace", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Stanley Sun",          "School": "Northeastern", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Stephanie Laplante",   "School": "University of Massachusetts", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Stephen Scopa",        "School": "Brown", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Saxon Hart",           "School": "William and Mary", "Initial Year": 2019, "Last Year Seen": 2022},
    {"Debater": "Sanket Bhalotia",      "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2025},
    {"Debater": "Sarah Barkatz",        "School": "CUNY", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Vishakh Sandwar",      "School": "NYU", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Travis Hunsberger",    "School": "University of Pittsburgh", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Taylor Small",         "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Talia Yett",           "School": "Brown", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sándor Lorange",       "School": "Tufts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Supriyaa Hejib",       "School": "University of Massachusetts", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Sunint Bindra",        "School": "Dartmouth", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Summit Sarkar",        "School": "Amherst", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Sudhish Rao",          "School": "Johns Hopkins", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Steven Macawili",      "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Steven Lou",           "School": "Princeton", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Stacy Amoako",         "School": "University of Massachusetts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Srivatsav Pyda",       "School": "Columbia", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Sree Dharmaraj",       "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sravya Dontharaju",    "School": "Tufts", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Sophie Fetter",        "School": "Villanova", "Initial Year": 2021, "Last Year Seen": 2024},
    {"Debater": "Sophia Marmai",        "School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sophia Farinella",     "School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2024},
    {"Debater": "Sophia Anderson",      "School": "University of Delaware", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sonam Tenzin",         "School": "Yale", "Initial Year": 2021, "Last Year Seen": 2023},
    {"Debater": "Sofia Little",         "School": "Rutgers", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Skyler Goldberg",      "School": "Tufts", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Skylar Jones",         "School": "NYU", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Simran Singh",         "School": "Brown", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sierra Maciorowski",   "School": "Stanford", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Siddhant Moily",       "School": "Brandeis", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Sidd Jain",            "School": "University of Chicago", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Sid Gupta",            "School": "Maryland", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Sia Kothari",          "School": "Bentley University", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Shriya Sane",          "School": "Penn", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Skyler Lee",           "School": "Amherst", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Shriya Kosuru",        "School": "William and Mary", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Shreyam Misra",        "School": "University of Chicago", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Shlomo Grun",          "School": "CUNY", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Shiqi Peng",           "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sheoli Lele",          "School": "William and Mary", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Shayna Leng",          "School": "Harvard", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Shanyu Thibaut Juneja","School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Seth Choi",            "School": "Johns Hopkins", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Serena Salam",         "School": "Wellesley", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Senthil",              "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sefat Samee",          "School": "Odette", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Sebastien Ludwig",     "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Sebastian Pollock",    "School": "Amherst", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sean Roth",            "School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Sean Oh",              "School": "Penn", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sean Nolan",           "School": "Northeastern", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Sean Berman",          "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Scarlett Wang",        "School": "Bates", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Sasya Koneru",         "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sasquatch Ray",        "School": "Harvard", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Sarwa Shah",           "School": "Wellesley", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Sare King",            "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sarah Hickey",         "School": "Wellesley", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Samyak Jain",          "School": "George Washington", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Samuel Rohwer",        "School": "Rutgers", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Samuel Butler",        "School": "University of Chicago", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Samra Lulseged",       "School": "Penn", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Samarth Jha",          "School": "Bates", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Sam Slack",            "School": "Bentley University", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Sam Siemer",           "School": "Johns Hopkins", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Sam Sagawa",           "School": "University of Chicago", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Sam Russo",            "School": "Tufts", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Sam Rohwer",           "School": "Rutgers", "Initial Year": 2019, "Last Year Seen": 2022},
    {"Debater": "Sam Passner",          "School": "University of Virginia", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Sam Mackin",           "School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Sajid Ibrahim",        "School": "Boston University", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Sabrina Yang",         "School": "NYU", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Sabrina Eager",        "School": "William and Mary", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ryan Yang",            "School": "MIT", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Ryan Flammer",         "School": "University of Virginia", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ryan Craig",           "School": "William and Mary", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Rushabh Patel",        "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Rush Patel",           "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Rui Gong",             "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Ross Khelemsky",       "School": "Penn", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Roshan Pillai",        "School": "University of Massachusetts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Roli Tinsley",         "School": "American", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Rohita Krishnakumar",  "School": "Maryland", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Rizky Ananda",         "School": "Penn", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Riya Bhattacharjee",   "School": "Wellesley", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Rishi Mukherjee",      "School": "University of Massachusetts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Rija Masroor",         "School": "William and Mary", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Richie Lu",            "School": "Columbia", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Richard Kim",          "School": "Yale", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Renee Wu",             "School": "Johns Hopkins", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Rene Garrett",         "School": "Denison", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Rebecca Mollet",       "School": "Maryland", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Rebecca Hsu",          "School": "University of Pittsburgh", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Rebeca Samano",        "School": "American", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Raymond Banke",        "School": "Columbia", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ray Yang",             "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Rana Ürek",            "School": "Columbia", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ramisa Rahman",        "School": "William and Mary", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Rainiero Disera",      "School": "Brandeis", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Ragy Amin",            "School": "University of Chicago", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Raghad Mohamed",       "School": "Bowdoin College", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Rafi Chowdhury",       "School": "William and Mary", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Rafay Abdul",          "School": "Bates", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Rachel Liu",           "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Qingxi Jaja Wang",     "School": "Johns Hopkins", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Pulin Wang",           "School": "Maryland", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Priyanka Mahat",       "School": "Brown", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Presley Forrest",      "School": "University of Massachusetts", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Pre Ferri",            "School": "Temple", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Pratham Lakhani",      "School": "Carnegie Mellon", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Prakhar Agrawal",      "School": "Amherst", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Prajata Roy",          "School": "NYU", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Pragnya Yerramalli",   "School": "Lehigh", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Peter Mao",            "School": "Johns Hopkins", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Peter Lawrence",       "School": "Maryland", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Pesandi Gunasekera",   "School": "University of Virginia", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Patrick Song",         "School": "Rutgers", "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Paneez Oilai",         "School": "Georgetown", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Owen Boice",           "School": "American", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Ovia Sundar",          "School": "Tufts", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Oscar Barrios",        "School": "Princeton", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Omogolo Pikinini",     "School": "Lehigh", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Omar Khan",            "School": "University of Chicago", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Olivia Taboada",       "School": "Temple", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Olivia Parashar",      "School": "Brandeis", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Olivia Mclane",        "School": "Temple", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Olivia Mastrangelo",   "School": "William and Mary", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Olivia Martinez",      "School": "Smith", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Olivia Lowry",         "School": "Johns Hopkins", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Olivia Ferrier",       "School": "University of Delaware", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Oliver Brazda",        "School": "Tufts", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Zach Fedyk",           "School": "University of Pittsburgh", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Winn Ryan",            "School": "Johns Hopkins", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Tyler Viljaste",       "School": "University of Pittsburgh", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Topher Zane",          "School": "William and Mary", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Shravani Subhedar",    "School": "Fordham", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Sebastian Rajguru",    "School": "William and Mary", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Ryan Shue",            "School": "William and Mary", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Ralph Montas Osias",   "School": "William and Mary", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Pierre Mathier",       "School": "Wesleyan", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Nusayba Chowdhury",    "School": "University of Pittsburgh", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Nora Sam",             "School": "Rutgers", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Noor Al-Saloum",       "School": "Johns Hopkins", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Noe Cifuentes",        "School": "Drexel", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Noah Ogata",           "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Noah Berman",          "School": "Northeastern", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Nitin Kumar",          "School": "Northeastern", "Initial Year": 2021, "Last Year Seen": 2023},
    {"Debater": "Nikola Simon",         "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Nikita Rodin",         "School": "University of Chicago", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Nikhil Rao",           "School": "William and Mary", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Nico Llorente Valin",  "School": "Fordham", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Nick Freilino",        "School": "Duquesne University", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Nick Claudio",         "School": "George Washington", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Nick Bukofsky",        "School": "Binghamton University", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Nicholas Sanchez",     "School": "Fordham", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Nicholas Perez",       "School": "Johns Hopkins", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Nicholas Kelly",       "School": "Harvard", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Nicholas Hao",         "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Nicholas Chuckas",     "School": "University of Virginia", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Nic Howayeck",         "School": "University of Massachusetts", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Nehemiah Cesar",       "School": "Williams", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Natt Sears",           "School": "Georgetown", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Nathaniel Maali-Pohl", "School": "University of Massachusetts", "Initial Year": 2026, "Last Year Seen": 2026},
    {"Debater": "Nathan Tang",          "School": "Bentley University", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Nathan Schechter",     "School": "Haverford", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Nathan Peng",          "School": "University of Chicago", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Nathan De Moura",      "School": "Harvard", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Natasha Raman",        "School": "Dartmouth", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Natalie George",       "School": "Odette", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Natalia Zorrilla",     "School": "Princeton", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Natalia Siwek",        "School": "Harvard", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Nadia Lee",            "School": "Maryland", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Nadeen Alomar",        "School": "Maryland", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Muskan Chhabra",       "School": "George Washington", "Initial Year": 2022, "Last Year Seen": 2022}, 
    {"Debater": "Muhammad Siddiqui",    "School": "NYU", "Initial Year": 2024, "Last Year Seen": 2025},
    {"Debater": "Muhammad Dhafer",      "School": "Stanford", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Mounisha Anumolu",     "School": "Dartmouth", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Monica Li",            "School": "Rutgers", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Mohammed Sarker",      "School": "Penn", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Mitchell Green",       "School": "Wesleyan", "Initial Year": 2020, "Last Year Seen": 2022},
    {"Debater": "Mireya Sanchez-Maes",  "School": "Harvard", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Mirada Makhmutova",    "School": "Boston University", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Miles Richardson",     "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Miles Gendebien",      "School": "Tufts", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Mido Sang",            "School": "University of Chicago", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Michelle Doan",        "School": "Amherst", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Michael Yoo",          "School": "Johns Hopkins", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Michael Tsutagawa",    "School": "Johns Hopkins", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Michael Tatum",        "School": "Boston College", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Michael Smith",        "School": "University of Delaware", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Michael Hong",         "School": "Northeastern", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Michael Clifford",     "School": "University of Massachusetts", "Initial Year": 2023, "Last Year Seen": 2024},
    {"Debater": "Michael Chen",         "School": "Penn", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Michael Carley",       "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Michael Avila",        "School": "Binghamton University", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Micah Kawecki",        "School": "University of Virginia", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Mia Scherer",          "School": "Wellesley", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Mia Madonna",          "School": "NYU", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Mia Kuchner",          "School": "American", "Initial Year": 2019, "Last Year Seen": 2021},
    {"Debater": "Mesoun Hassan",        "School": "American", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Mercer Mercer",        "School": "Rutgers", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Meghan Hendrix",       "School": "University of Chicago", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Megan Williams",       "School": "American", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Maxwell Weiner",       "School": "Brandeis", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Maxwell Guo",          "School": "NYU", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Max Zhou",             "School": "Hamilton", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Matthew Sinning",      "School": "Hamilton", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Matthew Li",           "School": "William and Mary", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Matt Kiley",           "School": "Harvard", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Matilda Stricherz",    "School": "Stanford", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Mary Wu",              "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2023},
    {"Debater": "Marjola Demollari",    "School": "University of Massachusetts", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Marissa Pereira",      "School": "University of Delaware", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Marina Pantner",       "School": "William and Mary", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Marilyn Santo",        "School": "Georgetown", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Mariam Nageeb",        "School": "Fordham", "Initial Year": 2023, "Last Year Seen": 2023},  
    {"Debater": "Maria Dubasov",        "School": "William and Mary", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Marcelo Rodriguez Parra","School": "Brown", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Marc Speeches",        "School": "Brown", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Marah Sami",           "School": "Amherst", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Manhua Kim",           "School": "Maryland", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Maksym Sherman",       "School": "Swarthmore", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Mai Al Shaaban",       "School": "Brandeis", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Mahir Abrar",          "School": "Lehigh", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Madeline Wyatt",       "School": "Columbia", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Madeline Turner",      "School": "Northeastern", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Madeleine Eichorn",    "School": "George Washington", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Mac O'Hara",           "School": "Penn", "Initial Year": 2019, "Last Year Seen": 2020},
    {"Debater": "Luke Chan",            "School": "Princeton", "Initial Year": 2021, "Last Year Seen": 2021},
    {"Debater": "Lukas Roybal",         "School": "Columbia", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Luka Bulic Braculj",   "School": "MIT", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Luis Luna",            "School": "Cornell", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Lucia Gonzalez",       "School": "Penn", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Lucas Irwin",          "School": "Princeton", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Luca Montoya",         "School": "Swarthmore", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Louis Mukama",         "School": "Harvard", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Lorenzo Songsare-Shevy","School": "Bates", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "London Cooper",        "School": "Columbia", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Logan Quick",          "School": "Harvard", "Initial Year": 2020, "Last Year Seen": 2022},   
    {"Debater": "Logan De Raspide Ross","School": "Boston University", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Lizzie Kerman",        "School": "William and Mary", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Liz Lin Moore",        "School": "Wellesley", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Liz Greer",            "School": "George Washington", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Linh Hoang",           "School": "Fordham", "Initial Year": 2025, "Last Year Seen": 2025},
    {"Debater": "Lindsey White",        "School": "University of North Carolina", "Initial Year": 2024, "Last Year Seen": 2024},
    {"Debater": "Lindsey Greenberg",    "School": "Northeastern", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Lindsay Khalluf",      "School": "Georgetown", "Initial Year": 2022, "Last Year Seen": 2022},
    {"Debater": "Lily Siru",            "School": "Haverford", "Initial Year": 2023, "Last Year Seen": 2022},
    {"Debater": "Lily Li",              "School": "Haverford", "Initial Year": 2020, "Last Year Seen": 2020},
    {"Debater": "Liam Rosario",         "School": "Binghamton", "Initial Year": 2019, "Last Year Seen": 2019},
    {"Debater": "Liam Bloom",           "School": "Wesleyan", "Initial Year": 2023, "Last Year Seen": 2023},
    {"Debater": "Lemuel Yu",            "School": "Boston University", "Initial Year": 2022, "Last Year Seen": 2022},   
]

recovered_debaters = pd.DataFrame(recovered_debaters)
valid_debaters = pd.read_csv("All_Debaters.csv")
valid_debaters = pd.concat([valid_debaters, recovered_debaters], ignore_index=True)
valid_debaters = valid_debaters.drop_duplicates()
valid_debaters.to_csv("All_Debaters.csv", index=False)

Minor parsing issues remained so the following pattern cleans for these remaining issues.

In [1584]:
unclean_patterns = [
    r"\*",
    r"\(R\)",
    r"\(",
    r"\)",
    r"She/Her",
    r"He/Him",
    r"They/She",
    r"He/They",
    r"They/Them",
]

To ensure no issues from timing of debaters, the following rechecks the first and last year the debaters were seen.

In [1585]:
df_clean_rounds = pd.read_csv("All_Rounds_With_Debaters_Index.csv")
df_clean_debaters = pd.read_csv("All_Debaters.csv")

name_cols = ["Speaker One Name", "Speaker Two Name", "Opponent One Name", "Opponent Two Name"]

long_df = pd.concat([
    df_clean_rounds[[col, "Season", "Year"]].rename(columns={col: "Debater"})
    for col in name_cols
], ignore_index=True).dropna(subset=["Debater"])

season_order = {"Fall": 0, "Spring": 1}  
long_df["_season_key"] = long_df["Year"] * 10 + long_df["Season"].map(season_order)

idx_first = long_df.groupby("Debater")["_season_key"].idxmin()
idx_last = long_df.groupby("Debater")["_season_key"].idxmax()

first_seen = long_df.loc[idx_first, ["Debater", "Year"]].rename(columns={"Year": "Computed Initial Year"})
last_seen = long_df.loc[idx_last, ["Debater", "Year"]].rename(columns={"Year": "Computed Final Year"})

first_last = pd.merge(first_seen, last_seen, on="Debater")

df_clean_debaters = pd.merge(df_clean_debaters, first_last, on="Debater", how="left")

df_clean_debaters["Initial Year"] = df_clean_debaters["Initial Year"].astype("Int64")
df_clean_debaters["Last Year Seen"] = df_clean_debaters["Last Year Seen"].astype("Int64")
df_clean_debaters["Computed Initial Year"] = df_clean_debaters["Computed Initial Year"].astype("Int64")
df_clean_debaters["Computed Final Year"] = df_clean_debaters["Computed Final Year"].astype("Int64")

df_clean_debaters["Initial Year Corrected"] = (
    df_clean_debaters["Computed Initial Year"].notna() &
    (df_clean_debaters["Computed Initial Year"] != df_clean_debaters["Initial Year"])
)
df_clean_debaters["Last Year Corrected"] = (
    df_clean_debaters["Computed Final Year"].notna() &
    (df_clean_debaters["Computed Final Year"] != df_clean_debaters["Last Year Seen"])
)

df_clean_debaters["Initial Year"] = df_clean_debaters["Initial Year"].where(
    ~df_clean_debaters["Initial Year Corrected"], df_clean_debaters["Computed Initial Year"]
)
df_clean_debaters["Last Year Seen"] = df_clean_debaters["Last Year Seen"].where(
    ~df_clean_debaters["Last Year Corrected"], df_clean_debaters["Computed Final Year"]
)

df_clean_debaters = df_clean_debaters.drop(
    columns=["Computed Initial Year", "Computed Final Year",
             "Initial Year Corrected", "Last Year Corrected"]
)

df_clean_debaters["Initial Year"] = df_clean_debaters["Initial Year"].astype("Int64")
df_clean_debaters["Last Year Seen"] = df_clean_debaters["Last Year Seen"].astype("Int64")
df_clean_debaters = df_clean_debaters.rename(columns={"Initial Year": "Initial Year Seen"})
df_clean_debaters = df_clean_debaters.rename(columns={"Last Year Seen": "Final Year Seen"})

df_clean_debaters.to_csv("All_Debaters_Tracked.csv", index=False)

The following code creates a table for all the rows that have identified judges.

In [1586]:
debate_rounds_df = pd.read_csv("Rounds.csv")
for pat in unclean_patterns:
        debate_rounds_df["Judge"] = debate_rounds_df["Judge"].str.replace(pat, "", regex=True)
debate_rounds_df["Judge"] = debate_rounds_df["Judge"].replace(COMMON_MISSPELLINGS)
valid_debaters = pd.read_csv("All_Debaters_Tracked.csv")

judge_school_df = debate_rounds_df.rename(columns={"Judge": "Debater"})
judge_school_df = judge_school_df.sort_values("Year")
valid_debaters_sorted = valid_debaters.sort_values("Initial Year Seen")

# first pass: match the affiliation that had already started by the round's year
judge_school_df = pd.merge_asof(
    judge_school_df,
    valid_debaters_sorted,
    left_on="Year",
    right_on="Initial Year Seen",
    by="Debater",
    direction="backward",
)

# second pass: for rows still unmatched, fall back to each debater's earliest known affiliation
earliest_debaters = (
    valid_debaters
    .sort_values("Initial Year Seen")
    .drop_duplicates(subset="Debater", keep="first")
    .set_index("Debater")
)

still_missing = judge_school_df["School"].isna()

judge_school_df.loc[still_missing, "School"] = (
    judge_school_df.loc[still_missing, "Debater"].map(earliest_debaters["School"])
)
judge_school_df.loc[still_missing, "Initial Year Seen"] = (
    judge_school_df.loc[still_missing, "Debater"].map(earliest_debaters["Initial Year Seen"])
)
judge_school_df.loc[still_missing, "Final Year Seen"] = (
    judge_school_df.loc[still_missing, "Debater"].map(earliest_debaters["Final Year Seen"])
)

judge_school_df = judge_school_df.rename(columns={
    "Debater": "Judge",
    "School": "Judge School",
    "Initial Year Seen": "Judge Initial Year",
    "Final Year Seen": "Judge Last Competed"
})

judge_school_df = judge_school_df[~judge_school_df["Judge School"].isna()]

judge_school_df.to_csv("All_Rounds_With_Judge.csv", index=False)

The following creates a table where all four debaters are identified as well as the judge.

In [1588]:
df = pd.read_csv("All_Rounds_With_Judge.csv")
speaker_cols = ["Speaker One Name", "Speaker Two Name", "Opponent One Name", "Opponent Two Name"]
for col in speaker_cols:
    for pat in unclean_patterns:
        df[col] = df[col].str.replace(pat, "", regex=True)
    df[col] = df[col].str.strip()

debate_roster = pd.read_csv("All_Debaters_Tracked.csv")

roster_lookup = debate_roster.drop_duplicates(subset="Debater").set_index("Debater")
valid_names = set(roster_lookup.index)

name_columns = {
    "Speaker One Name": "Speaker One",
    "Speaker Two Name": "Speaker Two",
    "Opponent One Name": "Opponent One",
    "Opponent Two Name": "Opponent Two",
}

lost_rounds = {}

for name_col, prefix in name_columns.items():
    cleaned = df[name_col].replace(COMMON_MISSPELLINGS)

    unmatched_mask = ~cleaned.isin(valid_names)
    lost_rounds[prefix] = cleaned[unmatched_mask].value_counts()

    df[name_col] = cleaned
    df[f"{prefix} School"] = cleaned.map(roster_lookup["School"])
    df[f"{prefix} Initial Year"] = cleaned.map(roster_lookup["Initial Year Seen"])
    df[f"{prefix} Last Competed"] = cleaned.map(roster_lookup["Final Year Seen"])


indiv_speaker_s1 = lost_rounds["Speaker One"]
indiv_speaker_s2 = lost_rounds["Speaker Two"]
indiv_speaker_o1 = lost_rounds["Opponent One"]
indiv_speaker_o2 = lost_rounds["Opponent Two"]

df_speaker_ids = df.copy()

school_cols = ["Speaker One School", "Speaker Two School",
               "Opponent One School", "Opponent Two School"]
df_speaker_ids = df_speaker_ids.dropna(subset=school_cols)

df_speaker_ids["Speaker Hybrid Status"] = df_speaker_ids["Speaker One School"] != df_speaker_ids["Speaker Two School"]
df_speaker_ids["Opponent Hybrid Status"] = df_speaker_ids["Opponent One School"] != df_speaker_ids["Opponent Two School"]

df_speaker_ids.to_csv("All_Rounds_With_Debaters_Index.csv", index=False)

combined = []
for prefix, counts in lost_rounds.items():
    temp = counts.rename("Rounds Lost").rename_axis("Name").reset_index()
    temp["Role"] = prefix
    combined.append(temp)

lost_rounds_all = pd.concat(combined, ignore_index=True)
lost_rounds_all = lost_rounds_all[["Role", "Name", "Rounds Lost"]]
lost_rounds_all.to_csv("lost_rounds_all.csv", index=False)

The above table creating new debater entries was created by examining the following csvs created.

In [1590]:
partner_pairs = []
partner_pairs.append(df[["Speaker One Name", "Speaker Two School", "Year"]]
                      .rename(columns={"Speaker One Name": "Debater", "Speaker Two School": "Partner School"}))
partner_pairs.append(df[["Speaker Two Name", "Speaker One School", "Year"]]
                      .rename(columns={"Speaker Two Name": "Debater", "Speaker One School": "Partner School"}))
partner_pairs.append(df[["Opponent One Name", "Opponent Two School", "Year"]]
                      .rename(columns={"Opponent One Name": "Debater", "Opponent Two School": "Partner School"}))
partner_pairs.append(df[["Opponent Two Name", "Opponent One School", "Year"]]
                      .rename(columns={"Opponent Two Name": "Debater", "Opponent One School": "Partner School"}))

all_partner_pairs = pd.concat(partner_pairs, ignore_index=True).dropna(subset=["Debater", "Partner School"])

all_partner_pairs_unmatched = all_partner_pairs[
    ~all_partner_pairs["Debater"].isin(valid_names)
].copy()

partner_school_summary_unmatched = (
    all_partner_pairs_unmatched.groupby("Debater")["Partner School"]
    .agg(lambda schools: sorted(set(schools)))
    .reset_index()
    .rename(columns={"Partner School": "Distinct Partner Schools"})
)
partner_school_summary_unmatched["Num Distinct Partner Schools"] = (
    partner_school_summary_unmatched["Distinct Partner Schools"].apply(len)
)

year_range_unmatched = (
    all_partner_pairs_unmatched.groupby("Debater")["Year"]
    .agg(First_Year_Seen="min", Last_Year_Seen="max")
    .reset_index()
)

partner_school_summary_unmatched = pd.merge(
    partner_school_summary_unmatched, year_range_unmatched, on="Debater", how="left"
)
partner_school_summary_unmatched = partner_school_summary_unmatched.sort_values(
    "Num Distinct Partner Schools", ascending=False
)
partner_school_summary_unmatched.to_csv("partner_school_summary_unmatched.csv", index=False)

partner_school_counts_unmatched = (
    all_partner_pairs_unmatched.groupby(["Debater", "Partner School"])
    .agg(Rounds=("Year", "size"), First_Year_Seen=("Year", "min"), Last_Year_Seen=("Year", "max"))
    .reset_index()
    .sort_values(["Debater", "Rounds"], ascending=[True, False])
)
partner_school_counts_unmatched.to_csv("partner_school_counts_unmatched.csv", index=False)

In [1269]:
public_private = [
    ['American', 'Private', 'South', 'No'],
    #['Adelphi', 'Private', 'North', 'No'],
    ['Amherst', 'Private', 'North', 'No'],
    #['Bard', 'Private', 'North', 'No'],
    ['Bates', 'Private', 'North', 'No'],
    ['Bentley University', 'Private', 'North', 'No'],
    #['Berkeley', 'Public', '', 'No'],
    #['Bard Graduate Center', 'Private', 'North', 'No'],
    ['Binghamton University', 'Public', 'North', 'No'],
    ['Boston College', 'Private', 'North', 'No'],
    ['Boston University', 'Private', 'North', 'No'], 
    #['Bowdoin College', 'Private', 'North', 'No'],
    ['Brierley Price Prior', 'Private', 'International', 'No'],
    #['Bradley', 'Private', '', 'No'],
    ['Brandeis', 'Private', 'North', 'No'],
    ['Brown', 'Private', 'North', 'Yes'],
    ['Bryn Mawr', 'Private', 'Central', 'No'], 
    #['Bucknell', '', '', 'No'],
    #['Cambridge', '', '', 'No'],
    #['Carleton', '', '', ''],
    #['Carnegie Mellon', '', 'Central', 'No'], 
    #['City College of San Francisco', '', '', 'No'], 
    #['Claremont', '', '', 'No'],
    #['Colgate', '', '', 'No'], 
    #['Columbia', '', '', 'Yes'], 
    #['Columbia Law', '', '', 'Yes'], 
    #['Cornell', '', '', 'Yes'], 
    #['CUNY', '', 'North', 'No'],
    #['Dalhousie', '', '', 'No'], 
    #['Dartmouth', '', '', 'Yes'], 
    #['Davidson', '', '', 'No'], 
    #['Denison', '', '', 'No'], 
    #['Drexel', '', '', 'No'], 
    #['Duke', '', '', 'No'],
    #['Duquesne University', '', '', 'No'], 
    #['Durham', '', '', 'No'], 
    #['Emmanuel', '', '', 'No'], 
    #['Emory', '', '', 'No'], 
    #['Fairfield', '', '', 'No'],
    #['Fisher College', '', '', 'No'], 
    #['Florida International University', '', '', 'No'],
    #['Florida State University', '', '', 'No'], 
    #['Fordham', '', '', ''], 
    #['FranklinandMarshall', '', '', ''],
    #['George Mason', '', '', ''], 
    ['Georgetown', 'Private', 'South', 'No'], 
    ['George Washington', 'Private', 'South', 'No'], 
    #['Glasgow', '', '', ''],
    #['Grinnell', '', '', ''], 
    #['Grove City College', '', '', ''], 
    #['Hamilton', '', '', ''], 
    #['Hart House', '', '', ''],
    #['Harvard', '', '', 'Yes'],
    #['Harvard Law', '', '', 'Yes'], 
    #['Haverford', '', '', ''], 
    #['HWS', '', '', ''],
    #['Hobart and William Smith', '', '', ''], 
    #['University of Dhaka', '', '', ''], 
    #['IIUM', '', '', ''],
    ['Johns Hopkins', 'Private', 'South', 'No'], 
    #['Kings', '', '', ''],
    #['Kwame Nkrumah University of Science and Technology', '', '', ''], 
    #['La Verne', '', '', ''],
    #['Lehigh', '', '', ''], 
    #['Loyola Marymount', '', '', ''], 
    #['Loyola University Chicago', '', '', ''],
    ['Maryland', 'Public', 'South', 'No'], 
    #['McGill', '', '', ''], 
    #['Middlebury', '', '', ''], 
    #['MIT', '', '', ''], 
    #['Moody Bible Institute', '', '', ''],
    #['Morehouse', '', '', ''], 
    #['Morehouse College', '', '', ''], 
    #['Mount Holyoke', '', '', ''], 
    #['Northeastern', '', '', ''],
    #['Northwestern', '', '', ''], 
    #['Notre Dame', '', '', ''], 
    #['NYU', 'Odette', '', '', ''], 
    #['Ottawa', '', '', ''], 
    #['Oxford', '', '', ''],
    #['Pace', '', '', ''], 
    #['Patrick Henry', '', '', ''], 
    #['Penn', '', '', 'Yes'],
    #["Prince George's Community College", '', '', ''], 
    #['Princeton', '', '', 'Yes'],
    #['Providence College', '', '', ''], 
    #['Quakers', '', '', ''], 
    #["Queen's University", '', '', ''], 
    #['RIT', '', '', ''],
    #['Rochester', '', '', ''], 
    #['RPI', '', '', ''], 
    #['Rutgers', '', '', ''], 
    #['San Jose State', '', '', ''], 
    #['Santa Clara', '', '', ''],
    #['Simon Fraser University', '', '', ''], 
    #["Simon's Rock College", '', '', ''], 
    #['Skidmore', '', '', ''],
    #['Smith', '', '', ''], 
    #['South Carolina', '', '', ''], 
    #['Spelman', '', '', ''], 
    #['St. Andrews', '', '', ''],
    #['Stanford', '', '', ''],
    #['St. Johns', '', '', ''], 
    #["St. Mary's", '', '', ''], 
    #['Stony Brook University', '', '', ''], 
    #['Swarthmore', '', '', ''],
    #['Sydney', '', '', ''], 
    #['Syracuse', '', '', ''], 
    #['Tal Aviv', '', '', ''], 
    #['Temple', '', '', ''], 
    #['TESU', '', '', ''],
    #['The College of New Jersey', '', '', ''], 
    #['Trinity', '', '', ''], 
    #['Tufts', '', '', ''], 
    #['Tulane', '', '', ''], 
    #['Tulsa', '', '', ''],
    #['UCD L&H', '', '', ''], 
    #['UCLA', '', '', ''], 
    #['UConn', '', '', ''], 
    #['UMBC', '', '', ''],
    #['University of Alaska Anchorage', '', '', ''], 
    #['University of Albany', '', '', ''],
    #['University of British Columbia', '', '', ''], 
    #['University of Calgary', '', '', ''],
    #['University of Chicago', '', '', ''], 
    #['University of Delaware', '', '', ''],
    #['University of Denver', '', '', ''], 
    #['University of Guelph', '', '', ''],
    #['University of Hawaii at Manoa', '', '', ''], 
    #['University of Massachusetts', '', '', ''],
    #['University of Michigan', '', '', ''], 
    #['University of Minnesota', '', '', ''],
    #['University of New South Wales', '', '', ''], 
    #['University of North Carolina', '', '', ''],
    #['University of Pittsburgh', '', '', ''],
    #['University of Southern California', '', '', ''],
    #['University of Sydney', '', '', ''], 
    #['University of the People', '', '', ''],
    #['University of Vermont', '', '', ''], 
    #['University of Virginia', '', '', ''],
    #['University of Waterloo', '', '', ''], 
    #['UT Austin', '', '', ''], 
    #['Vassar', '', '', ''], 
    #['Villanova', '', '', ''],
    #['Washington University in St. Louis', '', '', ''],
    #['Wellesley', '', '', ''], 
    #['Wesleyan', '', '', ''],
    #['Western', '', '', ''], 
    #['Wilfred Laurier University', '', '', ''],
    ['William and Mary', 'Public', 'South', 'No'],
    #['Williams', '', '', ''], 
    #['WSCC', '', '', ''], 
    #['York University', '', '', '', 
    ['Yale', 'Private', 'North', 'Yes']
]
public_private_df = pd.DataFrame(public_private, columns=['Judge School', 'Judge School Distinction', 'Judge School Region', 'Judge Ivy Distinction'])
judge_school_df = pd.merge(judge_school_df, public_private_df, how="left")
# Reorder for clarity
new_order = ['Year', 'Season', 'Tournament', 'Round', 'G/O', 'W/L', 'Speaker One Name', 'Speaker One Speaks', 'Speaker One Rank', 
             'Speaker Two Name', 'Speaker Two Speaks', 'Speaker Two Rank', 'Opponent One Name', 'Opponent One Speaks', 
             'Opponent One Rank', 'Opponent Two Name', 'Opponent Two Speaks', 'Opponent Two Rank', 'Judge', 'Judge School', 
             'Judge School Distinction', 'Judge Ivy Distinction', 'Judge School Region', 'Judge Initial Year', 'Judge Last Competed']
judge_school_df = judge_school_df.reindex(columns=new_order)
judge_school_df["Judge Initial Year"] = judge_school_df["Judge Initial Year"].astype(int)
judge_school_df["Judge Last Competed"] = judge_school_df["Judge Last Competed"].astype(int)
judge_school_df.to_csv("All_Rounds_With_Judge.csv", index=False)